In [ ]:
!pip install arch statsmodels skimpy mlflow polars xgboost lightgbm catboost duckdb

In [ ]:
import pandas as pd
import polars as pl
import polars.selectors as cs
import numpy as np
import duckdb
import skimpy as sk
import matplotlib.pyplot as plt
import seaborn as sns
import os
import shutil
import itertools
import glob
import mlflow
import mlflow.sklearn

from statsmodels.tsa.regime_switching.markov_regression import MarkovRegression
from arch import arch_model
from statsmodels.tsa.holtwinters import Holt
from statsmodels.tsa.seasonal import seasonal_decompose
from statsmodels.tsa.statespace.sarimax import SARIMAX
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.pipeline import Pipeline
from sklearn.metrics import *
from sklearn.preprocessing import *
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import Ridge, Lasso, ElasticNet
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import VotingRegressor, AdaBoostRegressor, ExtraTreesRegressor
from sklearn.model_selection import BaseCrossValidator
from sklearn.base import clone
from xgboost import XGBRegressor
from lightgbm import LGBMRegressor
from catboost import CatBoostRegressor

import joblib

In [ ]:
con = duckdb.connect()

print(
    con.execute("""
        DESCRIBE
        SELECT *
        FROM read_parquet('/content/drive/MyDrive/prc-2026-datasets/training_*.parquet')
    """).fetchdf()
)

con.execute("""
    COPY (
        SELECT
            MVT_ID_mvt,
            FLIGHT_ID_mvt,
            FLIGHT_mvt,
            FLIGHT_RULE_mvt,
            ADEP_mvt,
            ADES_mvt,
            PHASE_mvt,
            MVT_TIME_UTC_mvt,
            BLOCK_TIME_UTC_mvt,
            SCHED_TIME_UTC_mvt,
            AIRCRAFT_TYPE_mvt,
            RUNWAY_mvt,
            STAND_mvt,
            TAXITIME_SEC_mvt
        FROM read_parquet('/content/drive/MyDrive/prc-2026-datasets/training_*.parquet')
        WHERE PHASE_mvt = 'DEP'
    )
    TO '/content/drive/MyDrive/prc-2026-datasets/taxi_departures.parquet'
    (FORMAT PARQUET, COMPRESSION ZSTD)
""")

              column_name               column_type null   key default extra
0              MVT_ID_mvt                    DOUBLE  YES  None    None  None
1           FLIGHT_ID_mvt                    DOUBLE  YES  None    None  None
2              FLIGHT_mvt                   VARCHAR  YES  None    None  None
3         FLIGHT_RULE_mvt                   VARCHAR  YES  None    None  None
4                ADEP_mvt                   VARCHAR  YES  None    None  None
5                ADES_mvt                   VARCHAR  YES  None    None  None
6               PHASE_mvt                   VARCHAR  YES  None    None  None
7        MVT_TIME_UTC_mvt  TIMESTAMP WITH TIME ZONE  YES  None    None  None
8      BLOCK_TIME_UTC_mvt  TIMESTAMP WITH TIME ZONE  YES  None    None  None
9      SCHED_TIME_UTC_mvt  TIMESTAMP WITH TIME ZONE  YES  None    None  None
10      AIRCRAFT_TYPE_mvt                   VARCHAR  YES  None    None  None
11             RUNWAY_mvt                   VARCHAR  YES  None    None  None

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

In [ ]:
df = (
    pl.scan_parquet("/content/drive/MyDrive/prc-2026-datasets/taxi_departures.parquet")
    .collect()
)

print(df.shape)
print(df.schema)

(2085047, 14)
Schema({'MVT_ID_mvt': Float64, 'FLIGHT_ID_mvt': Float64, 'FLIGHT_mvt': String, 'FLIGHT_RULE_mvt': String, 'ADEP_mvt': String, 'ADES_mvt': String, 'PHASE_mvt': String, 'MVT_TIME_UTC_mvt': Datetime(time_unit='us', time_zone='UTC'), 'BLOCK_TIME_UTC_mvt': Datetime(time_unit='us', time_zone='UTC'), 'SCHED_TIME_UTC_mvt': Datetime(time_unit='us', time_zone='UTC'), 'AIRCRAFT_TYPE_mvt': String, 'RUNWAY_mvt': String, 'STAND_mvt': String, 'TAXITIME_SEC_mvt': Int32})


In [ ]:
df

MVT_ID_mvt,FLIGHT_ID_mvt,FLIGHT_mvt,FLIGHT_RULE_mvt,ADEP_mvt,ADES_mvt,PHASE_mvt,MVT_TIME_UTC_mvt,BLOCK_TIME_UTC_mvt,SCHED_TIME_UTC_mvt,AIRCRAFT_TYPE_mvt,RUNWAY_mvt,STAND_mvt,TAXITIME_SEC_mvt
f64,f64,str,str,str,str,str,"datetime[μs, UTC]","datetime[μs, UTC]","datetime[μs, UTC]",str,str,str,i32
1.82234017e8,2.80602939e8,"""LH982""","""I""","""EDDF""","""EIDW""","""DEP""",2025-01-07 21:10:06 UTC,2025-01-07 21:00:11 UTC,2025-01-07 21:00:00 UTC,"""A320""","""25C""","""B41""",595
1.82234011e8,2.80602803e8,"""LH796""","""I""","""EDDF""","""VHHH""","""DEP""",2025-01-07 21:11:23 UTC,2025-01-07 20:53:18 UTC,2025-01-07 20:50:00 UTC,"""A346""","""18""","""A20""",1085
1.82234021e8,2.8060292e8,"""LH958""","""I""","""EDDF""","""EGBB""","""DEP""",2025-01-07 21:12:29 UTC,2025-01-07 21:01:01 UTC,2025-01-07 21:00:00 UTC,"""A320""","""25C""","""B48""",688
1.82234008e8,2.80602701e8,"""LH778""","""I""","""EDDF""","""WSSS""","""DEP""",2025-01-07 21:14:07 UTC,2025-01-07 20:59:29 UTC,2025-01-07 20:45:00 UTC,"""B748""","""18""","""A69""",878
1.82234019e8,2.80602945e8,"""LH948""","""I""","""EDDF""","""EGCC""","""DEP""",2025-01-07 21:16:34 UTC,2025-01-07 21:02:36 UTC,2025-01-07 21:00:00 UTC,"""A320""","""25C""","""V119""",838
…,…,…,…,…,…,…,…,…,…,…,…,…,…
2.01598814e8,2.94561204e8,"""SVA278""","""I""","""LTFM""","""OEJN""","""DEP""",2025-12-31 16:00:00 UTC,2025-12-31 15:30:00 UTC,2025-12-31 15:25:00 UTC,"""A21N""","""35L""","""B3L""",1800
2.01598826e8,2.94561612e8,"""THY6JG""","""I""","""LTFM""","""LTDA""","""DEP""",2025-12-31 15:59:58 UTC,2025-12-31 15:46:01 UTC,2025-12-31 15:45:00 UTC,"""A319""","""36""","""113""",837
2.01598817e8,2.94561588e8,"""PBD862""","""I""","""LTFM""","""URMO""","""DEP""",2025-12-31 16:00:56 UTC,2025-12-31 15:33:03 UTC,2025-12-31 13:00:00 UTC,"""B738""","""35L""","""A3R""",1673


## EDA

In [ ]:
q = (
    pl.scan_parquet("/content/drive/MyDrive/prc-2026-datasets/taxi_departures.parquet")
    .select("TAXITIME_SEC_mvt")
    .filter(pl.col("TAXITIME_SEC_mvt").is_not_null())
)

target_stats = q.select([
    pl.len().alias("n"),
    pl.col("TAXITIME_SEC_mvt").min().alias("min"),
    pl.col("TAXITIME_SEC_mvt").quantile(0.001).alias("q0.1%"),
    pl.col("TAXITIME_SEC_mvt").quantile(0.01).alias("q1%"),
    pl.col("TAXITIME_SEC_mvt").quantile(0.05).alias("q5%"),
    pl.col("TAXITIME_SEC_mvt").quantile(0.10).alias("q10%"),
    pl.col("TAXITIME_SEC_mvt").quantile(0.25).alias("q25%"),
    pl.col("TAXITIME_SEC_mvt").median().alias("q50%"),
    pl.col("TAXITIME_SEC_mvt").quantile(0.75).alias("q75%"),
    pl.col("TAXITIME_SEC_mvt").quantile(0.90).alias("q90%"),
    pl.col("TAXITIME_SEC_mvt").quantile(0.95).alias("q95%"),
    pl.col("TAXITIME_SEC_mvt").quantile(0.99).alias("q99%"),
    pl.col("TAXITIME_SEC_mvt").quantile(0.999).alias("q99.9%"),
    pl.col("TAXITIME_SEC_mvt").max().alias("max"),
    pl.col("TAXITIME_SEC_mvt").mean().alias("mean"),
    pl.col("TAXITIME_SEC_mvt").std().alias("std"),
]).collect()

print(target_stats)

shape: (1, 16)
┌─────────┬─────┬───────┬───────┬───┬────────┬────────┬────────────┬────────────┐
│ n       ┆ min ┆ q0.1% ┆ q1%   ┆ … ┆ q99.9% ┆ max    ┆ mean       ┆ std        │
│ ---     ┆ --- ┆ ---   ┆ ---   ┆   ┆ ---    ┆ ---    ┆ ---        ┆ ---        │
│ u32     ┆ i32 ┆ f64   ┆ f64   ┆   ┆ f64    ┆ i32    ┆ f64        ┆ f64        │
╞═════════╪═════╪═══════╪═══════╪═══╪════════╪════════╪════════════╪════════════╡
│ 2085047 ┆ -12 ┆ 119.0 ┆ 291.0 ┆ … ┆ 4501.0 ┆ 131167 ┆ 991.200707 ┆ 546.359769 │
└─────────┴─────┴───────┴───────┴───┴────────┴────────┴────────────┴────────────┘


In [ ]:
q.select([
    (pl.col("TAXITIME_SEC_mvt") < 0).sum().alias("negative"),
    (pl.col("TAXITIME_SEC_mvt") == 0).sum().alias("zero"),
    (pl.col("TAXITIME_SEC_mvt") < 60).sum().alias("under_1min"),
    (pl.col("TAXITIME_SEC_mvt") > 3600).sum().alias("over_1hr"),
    (pl.col("TAXITIME_SEC_mvt") > 7200).sum().alias("over_2hr"),
]).collect()

negative,zero,under_1min,over_1hr,over_2hr
u32,u32,u32,u32,u32
369,19,1105,4126,584


In [ ]:
bins = q.collect()

quantile_bins = bins.select(
    pl.col("TAXITIME_SEC_mvt").qcut(
        [0.01, 0.05, 0.10, 0.25, 0.50, 0.75, 0.90, 0.95, 0.99],
        labels=[
            "Q0-Q1",
            "Q1-Q5",
            "Q5-Q10",
            "Q10-Q25",
            "Q25-Q50",
            "Q50-Q75",
            "Q75-Q90",
            "Q90-Q95",
            "Q95-Q99",
            "Q99-Q100"
        ]
    ).alias("quantile")
).group_by("quantile").len()

print(quantile_bins)

shape: (10, 2)
┌──────────┬────────┐
│ quantile ┆ len    │
│ ---      ┆ ---    │
│ cat      ┆ u32    │
╞══════════╪════════╡
│ Q10-Q25  ┆ 313044 │
│ Q50-Q75  ┆ 519893 │
│ Q5-Q10   ┆ 104236 │
│ Q95-Q99  ┆ 83193  │
│ Q0-Q1    ┆ 20861  │
│ Q99-Q100 ┆ 20834  │
│ Q25-Q50  ┆ 523041 │
│ Q75-Q90  ┆ 311526 │
│ Q1-Q5    ┆ 84060  │
│ Q90-Q95  ┆ 104359 │
└──────────┴────────┘


In [ ]:
airport_stats = (
    pl.scan_parquet("/content/drive/MyDrive/prc-2026-datasets/taxi_departures.parquet")
    .group_by("ADEP_mvt")
    .agg([
        pl.len().alias("n"),
        pl.col("TAXITIME_SEC_mvt").mean().alias("mean"),
        pl.col("TAXITIME_SEC_mvt").median().alias("median"),
        pl.col("TAXITIME_SEC_mvt").std().alias("std"),
        pl.col("TAXITIME_SEC_mvt").quantile(0.10).alias("q10"),
        pl.col("TAXITIME_SEC_mvt").quantile(0.25).alias("q25"),
        pl.col("TAXITIME_SEC_mvt").quantile(0.75).alias("q75"),
        pl.col("TAXITIME_SEC_mvt").quantile(0.90).alias("q90"),
        pl.col("TAXITIME_SEC_mvt").quantile(0.95).alias("q95"),
        pl.col("TAXITIME_SEC_mvt").quantile(0.99).alias("q99"),
        pl.col("TAXITIME_SEC_mvt").min().alias("min"),
        pl.col("TAXITIME_SEC_mvt").max().alias("max"),
    ])
    .sort("n", descending=True)
    .collect()
)

print(airport_stats)

shape: (10, 13)
┌──────────┬────────┬─────────────┬────────┬───┬────────┬────────┬─────┬────────┐
│ ADEP_mvt ┆ n      ┆ mean        ┆ median ┆ … ┆ q95    ┆ q99    ┆ min ┆ max    │
│ ---      ┆ ---    ┆ ---         ┆ ---    ┆   ┆ ---    ┆ ---    ┆ --- ┆ ---    │
│ str      ┆ u32    ┆ f64         ┆ f64    ┆   ┆ f64    ┆ f64    ┆ i32 ┆ i32    │
╞══════════╪════════╪═════════════╪════════╪═══╪════════╪════════╪═════╪════════╡
│ LTFM     ┆ 272965 ┆ 1052.952269 ┆ 963.0  ┆ … ┆ 1749.0 ┆ 2587.0 ┆ -12 ┆ 10309  │
│ EHAM     ┆ 247951 ┆ 784.559748  ┆ 742.0  ┆ … ┆ 1309.0 ┆ 1775.0 ┆ -4  ┆ 11032  │
│ LFPG     ┆ 239552 ┆ 1019.52331  ┆ 954.0  ┆ … ┆ 1681.0 ┆ 2409.0 ┆ -12 ┆ 84240  │
│ EGLL     ┆ 239546 ┆ 1364.239319 ┆ 1319.0 ┆ … ┆ 1989.0 ┆ 2701.0 ┆ 9   ┆ 11452  │
│ EDDF     ┆ 230141 ┆ 863.478155  ┆ 837.0  ┆ … ┆ 1399.0 ┆ 1819.0 ┆ 2   ┆ 6907   │
│ LEMD     ┆ 212242 ┆ 1014.709982 ┆ 985.0  ┆ … ┆ 1559.0 ┆ 1932.0 ┆ 39  ┆ 3598   │
│ LEBL     ┆ 179705 ┆ 956.549356  ┆ 906.0  ┆ … ┆ 1501.0 ┆ 1962.0 ┆ -8  ┆ 3604   │


In [ ]:
time_stats = (
    pl.scan_parquet("/content/drive/MyDrive/prc-2026-datasets/taxi_departures.parquet")
    .with_columns([
        pl.col("BLOCK_TIME_UTC_mvt").dt.hour().alias("utc_hour"),
        pl.col("BLOCK_TIME_UTC_mvt").dt.weekday().alias("weekday"),
    ])
    .group_by("utc_hour")
    .agg([
        pl.len().alias("n"),
        pl.col("TAXITIME_SEC_mvt").mean().alias("mean"),
        pl.col("TAXITIME_SEC_mvt").median().alias("median"),
        pl.col("TAXITIME_SEC_mvt").quantile(0.90).alias("q90"),
        pl.col("TAXITIME_SEC_mvt").quantile(0.95).alias("q95"),
    ])
    .sort("utc_hour")
    .collect()
)

print(time_stats)

shape: (24, 6)
┌──────────┬────────┬─────────────┬────────┬────────┬────────┐
│ utc_hour ┆ n      ┆ mean        ┆ median ┆ q90    ┆ q95    │
│ ---      ┆ ---    ┆ ---         ┆ ---    ┆ ---    ┆ ---    │
│ i8       ┆ u32    ┆ f64         ┆ f64    ┆ f64    ┆ f64    │
╞══════════╪════════╪═════════════╪════════╪════════╪════════╡
│ 0        ┆ 4399   ┆ 926.889975  ┆ 844.0  ┆ 1270.0 ┆ 1489.0 │
│ 1        ┆ 5674   ┆ 1060.499648 ┆ 967.0  ┆ 1498.0 ┆ 1690.0 │
│ 2        ┆ 9300   ┆ 1046.336667 ┆ 987.0  ┆ 1505.0 ┆ 1739.0 │
│ 3        ┆ 24856  ┆ 942.734712  ┆ 849.0  ┆ 1328.0 ┆ 1570.0 │
│ 4        ┆ 59835  ┆ 906.488978  ┆ 845.0  ┆ 1314.0 ┆ 1505.0 │
│ …        ┆ …      ┆ …           ┆ …      ┆ …      ┆ …      │
│ 19       ┆ 117396 ┆ 969.812225  ┆ 901.0  ┆ 1432.0 ┆ 1621.0 │
│ 20       ┆ 100800 ┆ 990.284107  ┆ 917.0  ┆ 1451.0 ┆ 1674.0 │
│ 21       ┆ 45175  ┆ 1046.690647 ┆ 967.0  ┆ 1554.0 ┆ 1789.0 │
│ 22       ┆ 26498  ┆ 1063.059401 ┆ 967.0  ┆ 1501.0 ┆ 1789.0 │
│ 23       ┆ 12906  ┆ 1178.497753 ┆ 1026

In [ ]:
numeric = (
    pl.scan_parquet("/content/drive/MyDrive/prc-2026-datasets/taxi_departures.parquet")
    .with_columns([
        (
            (pl.col("MVT_TIME_UTC_mvt") -
             pl.col("BLOCK_TIME_UTC_mvt"))
            .dt.total_seconds()
        ).alias("observed_flight_time"),

        (
            (pl.col("BLOCK_TIME_UTC_mvt") -
             pl.col("SCHED_TIME_UTC_mvt"))
            .dt.total_seconds()
        ).alias("departure_delay"),

        pl.col("BLOCK_TIME_UTC_mvt").dt.hour().alias("utc_hour"),
        pl.col("BLOCK_TIME_UTC_mvt").dt.weekday().alias("weekday"),
    ])
    .select([
        "TAXITIME_SEC_mvt",
        "observed_flight_time",
        "departure_delay",
        "utc_hour",
        "weekday",
    ])
    .collect()
)

print(numeric.corr())

shape: (5, 5)
┌──────────────────┬──────────────────────┬─────────────────┬──────────┬──────────┐
│ TAXITIME_SEC_mvt ┆ observed_flight_time ┆ departure_delay ┆ utc_hour ┆ weekday  │
│ ---              ┆ ---                  ┆ ---             ┆ ---      ┆ ---      │
│ f64              ┆ f64                  ┆ f64             ┆ f64      ┆ f64      │
╞══════════════════╪══════════════════════╪═════════════════╪══════════╪══════════╡
│ 1.0              ┆ 1.0                  ┆ -0.005643       ┆ 0.017068 ┆ 0.008973 │
│ 1.0              ┆ 1.0                  ┆ -0.005643       ┆ 0.017068 ┆ 0.008973 │
│ -0.005643        ┆ -0.005643            ┆ 1.0             ┆ 0.084691 ┆ 0.021324 │
│ 0.017068         ┆ 0.017068             ┆ 0.084691        ┆ 1.0      ┆ 0.006567 │
│ 0.008973         ┆ 0.008973             ┆ 0.021324        ┆ 0.006567 ┆ 1.0      │
└──────────────────┴──────────────────────┴─────────────────┴──────────┴──────────┘


In [ ]:
bad = (
    pl.scan_parquet("/content/drive/MyDrive/prc-2026-datasets/taxi_departures.parquet")
    .filter(pl.col("TAXITIME_SEC_mvt") < 0)
    .collect()
)

print(bad)

shape: (369, 14)
┌───────────┬───────────┬───────────┬───────────┬───┬───────────┬───────────┬───────────┬──────────┐
│ MVT_ID_mv ┆ FLIGHT_ID ┆ FLIGHT_mv ┆ FLIGHT_RU ┆ … ┆ AIRCRAFT_ ┆ RUNWAY_mv ┆ STAND_mvt ┆ TAXITIME │
│ t         ┆ _mvt      ┆ t         ┆ LE_mvt    ┆   ┆ TYPE_mvt  ┆ t         ┆ ---       ┆ _SEC_mvt │
│ ---       ┆ ---       ┆ ---       ┆ ---       ┆   ┆ ---       ┆ ---       ┆ str       ┆ ---      │
│ f64       ┆ f64       ┆ str       ┆ str       ┆   ┆ str       ┆ str       ┆           ┆ i32      │
╞═══════════╪═══════════╪═══════════╪═══════════╪═══╪═══════════╪═══════════╪═══════════╪══════════╡
│ 1.8297711 ┆ 2.8086487 ┆ IBE2601   ┆ I         ┆ … ┆ A332      ┆ 06L       ┆ 214       ┆ -8       │
│ 4e8       ┆ 5e8       ┆           ┆           ┆   ┆           ┆           ┆           ┆          │
│ 1.8237934 ┆ 2.8046143 ┆ AFR15WK   ┆ I         ┆ … ┆ A318      ┆ 26R       ┆ null      ┆ -11      │
│ 2e8       ┆ e8        ┆           ┆           ┆   ┆           ┆         

In [ ]:
bad.select([
    "MVT_ID_mvt",
    "FLIGHT_mvt",
    "ADEP_mvt",
    "BLOCK_TIME_UTC_mvt",
    "MVT_TIME_UTC_mvt",
    "TAXITIME_SEC_mvt"
])

MVT_ID_mvt,FLIGHT_mvt,ADEP_mvt,BLOCK_TIME_UTC_mvt,MVT_TIME_UTC_mvt,TAXITIME_SEC_mvt
f64,str,str,"datetime[μs, UTC]","datetime[μs, UTC]",i32
1.82977114e8,"""IBE2601""","""LEBL""",2025-01-15 00:55:05 UTC,2025-01-15 00:54:57 UTC,-8
1.82379342e8,"""AFR15WK""","""LFPG""",2025-01-03 20:28:05 UTC,2025-01-03 20:27:54 UTC,-11
1.82363682e8,"""CAO3158""","""LFPG""",2025-01-14 11:27:04 UTC,2025-01-14 11:26:56 UTC,-8
1.8236476e8,"""SWT5745""","""LFPG""",2025-01-10 00:45:00 UTC,2025-01-10 00:44:59 UTC,-1
1.82374475e8,"""KLM84F""","""LFPG""",2025-01-08 08:59:01 UTC,2025-01-08 08:58:59 UTC,-2
…,…,…,…,…,…
2.01425471e8,"""ELY348""","""LSZH""",2025-12-31 12:51:03 UTC,2025-12-31 12:50:57 UTC,-6
2.01425774e8,"""HBZPF""","""LSZH""",2025-12-31 13:06:26 UTC,2025-12-31 13:06:16 UTC,-10
2.01424788e8,"""N149CK""","""LSZH""",2025-12-30 13:31:50 UTC,2025-12-30 13:31:49 UTC,-1


In [ ]:
extreme = (
    pl.scan_parquet("/content/drive/MyDrive/prc-2026-datasets/taxi_departures.parquet")
    .filter(
        (pl.col("TAXITIME_SEC_mvt") < 0) |
        (pl.col("TAXITIME_SEC_mvt") > 7200)
    )
    .select([
        "MVT_ID_mvt",
        "FLIGHT_ID_mvt",
        "FLIGHT_mvt",
        "ADEP_mvt",
        "ADES_mvt",
        "MVT_TIME_UTC_mvt",
        "BLOCK_TIME_UTC_mvt",
        "SCHED_TIME_UTC_mvt",
        "AIRCRAFT_TYPE_mvt",
        "RUNWAY_mvt",
        "STAND_mvt",
        "TAXITIME_SEC_mvt"
    ])
    .sort("TAXITIME_SEC_mvt", descending=True)
    .collect()
)

print(
    extreme.group_by("ADEP_mvt")
    .agg([
        pl.len().alias("n"),
        pl.col("TAXITIME_SEC_mvt").min().alias("min_TAXITIME_SEC_mvt"),
        pl.col("TAXITIME_SEC_mvt").median().alias("median_TAXITIME_SEC_mvt"),
        pl.col("TAXITIME_SEC_mvt").max().alias("max_TAXITIME_SEC_mvt")
    ])
    .sort("max_TAXITIME_SEC_mvt", descending=True)
)

shape: (7, 5)
┌──────────┬─────┬──────────────────────┬─────────────────────────┬──────────────────────┐
│ ADEP_mvt ┆ n   ┆ min_TAXITIME_SEC_mvt ┆ median_TAXITIME_SEC_mvt ┆ max_TAXITIME_SEC_mvt │
│ ---      ┆ --- ┆ ---                  ┆ ---                     ┆ ---                  │
│ str      ┆ u32 ┆ i32                  ┆ f64                     ┆ i32                  │
╞══════════╪═════╪══════════════════════╪═════════════════════════╪══════════════════════╡
│ LIRF     ┆ 480 ┆ 7202                 ┆ 9959.5                  ┆ 131167               │
│ LSZH     ┆ 304 ┆ -12                  ┆ -6.0                    ┆ 87341                │
│ LFPG     ┆ 98  ┆ -12                  ┆ -2.0                    ┆ 84240                │
│ EGLL     ┆ 37  ┆ 7255                 ┆ 8464.0                  ┆ 11452                │
│ EHAM     ┆ 6   ┆ -4                   ┆ 4374.5                  ┆ 11032                │
│ LTFM     ┆ 23  ┆ -12                  ┆ 7618.0                  ┆ 10309   

#Per Airport in every country

In [ ]:
airport_tz = {
    "EDDF": "Europe/Berlin",
    "EDDM": "Europe/Berlin",
    "EGLL": "Europe/London",
    "EHAM": "Europe/Amsterdam",
    "LEBL": "Europe/Madrid",
    "LEMD": "Europe/Madrid",
    "LFPG": "Europe/Paris",
    "LIRF": "Europe/Rome",
    "LTFM": "Europe/Istanbul",
    "LSZH": "Europe/Zurich",
}

lf = pl.scan_parquet("/content/drive/MyDrive/prc-2026-datasets/taxi_departures.parquet")

frames = []

for airport, timezone in airport_tz.items():

    tmp = (
        lf
        .filter(pl.col("ADEP_mvt") == airport)
        .with_columns(
            pl.col("BLOCK_TIME_UTC_mvt")
            .dt.convert_time_zone(timezone)
            .dt.replace_time_zone(None)
            .alias("local_datetime")
        )
        .with_columns([
            pl.col("local_datetime").dt.hour().alias("local_hour"),
            pl.col("local_datetime").dt.minute().alias("local_minute"),
            pl.col("local_datetime").dt.weekday().alias("local_weekday"),
            pl.col("local_datetime").dt.month().alias("local_month"),
            pl.col("local_datetime").dt.ordinal_day().alias("local_day"),
            (
                pl.col("local_datetime").dt.weekday() >= 6
            ).alias("is_weekend"),
        ])
        .select([
            "ADEP_mvt",
            "TAXITIME_SEC_mvt",
            "BLOCK_TIME_UTC_mvt",
            "local_datetime",
            "local_hour",
            "local_minute",
            "local_weekday",
            "local_month",
            "local_day",
            "is_weekend",
        ])
        .collect()
    )

    frames.append(tmp)

local_df = pl.concat(frames)

print(local_df.shape)
print(local_df.head())

(2085047, 10)
shape: (5, 10)
┌──────────┬───────────┬───────────┬───────────┬───┬───────────┬───────────┬───────────┬───────────┐
│ ADEP_mvt ┆ TAXITIME_ ┆ BLOCK_TIM ┆ local_dat ┆ … ┆ local_wee ┆ local_mon ┆ local_day ┆ is_weeken │
│ ---      ┆ SEC_mvt   ┆ E_UTC_mvt ┆ etime     ┆   ┆ kday      ┆ th        ┆ ---       ┆ d         │
│ str      ┆ ---       ┆ ---       ┆ ---       ┆   ┆ ---       ┆ ---       ┆ i16       ┆ ---       │
│          ┆ i32       ┆ datetime[ ┆ datetime[ ┆   ┆ i8        ┆ i8        ┆           ┆ bool      │
│          ┆           ┆ μs, UTC]  ┆ μs]       ┆   ┆           ┆           ┆           ┆           │
╞══════════╪═══════════╪═══════════╪═══════════╪═══╪═══════════╪═══════════╪═══════════╪═══════════╡
│ EDDF     ┆ 595       ┆ 2025-01-0 ┆ 2025-01-0 ┆ … ┆ 2         ┆ 1         ┆ 7         ┆ false     │
│          ┆           ┆ 7         ┆ 7         ┆   ┆           ┆           ┆           ┆           │
│          ┆           ┆ 21:00:11  ┆ 22:00:11  ┆   ┆          

In [ ]:
airport_hour = (
    local_df
    .group_by(["ADEP_mvt", "local_hour"])
    .agg([
        pl.len().alias("n"),

        pl.col("TAXITIME_SEC_mvt")
        .mean()
        .alias("mean"),

        pl.col("TAXITIME_SEC_mvt")
        .median()
        .alias("median"),

        pl.col("TAXITIME_SEC_mvt")
        .std()
        .alias("std"),

        pl.col("TAXITIME_SEC_mvt")
        .quantile(0.10)
        .alias("q10"),

        pl.col("TAXITIME_SEC_mvt")
        .quantile(0.25)
        .alias("q25"),

        pl.col("TAXITIME_SEC_mvt")
        .quantile(0.75)
        .alias("q75"),

        pl.col("TAXITIME_SEC_mvt")
        .quantile(0.90)
        .alias("q90"),

        pl.col("TAXITIME_SEC_mvt")
        .quantile(0.95)
        .alias("q95"),

        pl.col("TAXITIME_SEC_mvt")
        .quantile(0.99)
        .alias("q99"),
    ])
    .sort(["ADEP_mvt", "local_hour"])
)

print(airport_hour)

shape: (234, 12)
┌──────────┬────────────┬───────┬─────────────┬───┬────────┬────────┬────────┬────────┐
│ ADEP_mvt ┆ local_hour ┆ n     ┆ mean        ┆ … ┆ q75    ┆ q90    ┆ q95    ┆ q99    │
│ ---      ┆ ---        ┆ ---   ┆ ---         ┆   ┆ ---    ┆ ---    ┆ ---    ┆ ---    │
│ str      ┆ i8         ┆ u32   ┆ f64         ┆   ┆ f64    ┆ f64    ┆ f64    ┆ f64    │
╞══════════╪════════════╪═══════╪═════════════╪═══╪════════╪════════╪════════╪════════╡
│ EDDF     ┆ 0          ┆ 6     ┆ 429.666667  ┆ … ┆ 635.0  ┆ 650.0  ┆ 650.0  ┆ 650.0  │
│ EDDF     ┆ 1          ┆ 4     ┆ 471.25      ┆ … ┆ 533.0  ┆ 692.0  ┆ 692.0  ┆ 692.0  │
│ EDDF     ┆ 2          ┆ 2     ┆ 442.5       ┆ … ┆ 576.0  ┆ 576.0  ┆ 576.0  ┆ 576.0  │
│ EDDF     ┆ 3          ┆ 3     ┆ 382.333333  ┆ … ┆ 608.0  ┆ 608.0  ┆ 608.0  ┆ 608.0  │
│ EDDF     ┆ 4          ┆ 1339  ┆ 963.082151  ┆ … ┆ 1095.0 ┆ 1209.0 ┆ 1277.0 ┆ 1503.0 │
│ …        ┆ …          ┆ …     ┆ …           ┆ … ┆ …      ┆ …      ┆ …      ┆ …      │
│ LTFM     ┆ 19

In [ ]:
utc_hour = (
    pl.scan_parquet("/content/drive/MyDrive/prc-2026-datasets/taxi_departures.parquet")
    .with_columns(
        pl.col("BLOCK_TIME_UTC_mvt")
        .dt.hour()
        .alias("utc_hour")
    )
    .group_by("utc_hour")
    .agg([
        pl.len().alias("n"),
        pl.col("TAXITIME_SEC_mvt").median().alias("median"),
        pl.col("TAXITIME_SEC_mvt").mean().alias("mean"),
        pl.col("TAXITIME_SEC_mvt").quantile(0.90).alias("q90"),
        pl.col("TAXITIME_SEC_mvt").quantile(0.95).alias("q95"),
    ])
    .sort("utc_hour")
    .collect()
)

print(utc_hour)

shape: (24, 6)
┌──────────┬────────┬────────┬─────────────┬────────┬────────┐
│ utc_hour ┆ n      ┆ median ┆ mean        ┆ q90    ┆ q95    │
│ ---      ┆ ---    ┆ ---    ┆ ---         ┆ ---    ┆ ---    │
│ i8       ┆ u32    ┆ f64    ┆ f64         ┆ f64    ┆ f64    │
╞══════════╪════════╪════════╪═════════════╪════════╪════════╡
│ 0        ┆ 4399   ┆ 844.0  ┆ 926.889975  ┆ 1270.0 ┆ 1489.0 │
│ 1        ┆ 5674   ┆ 967.0  ┆ 1060.499648 ┆ 1498.0 ┆ 1690.0 │
│ 2        ┆ 9300   ┆ 987.0  ┆ 1046.336667 ┆ 1505.0 ┆ 1739.0 │
│ 3        ┆ 24856  ┆ 849.0  ┆ 942.734712  ┆ 1328.0 ┆ 1570.0 │
│ 4        ┆ 59835  ┆ 845.0  ┆ 906.488978  ┆ 1314.0 ┆ 1505.0 │
│ …        ┆ …      ┆ …      ┆ …           ┆ …      ┆ …      │
│ 19       ┆ 117396 ┆ 901.0  ┆ 969.812225  ┆ 1432.0 ┆ 1621.0 │
│ 20       ┆ 100800 ┆ 917.0  ┆ 990.284107  ┆ 1451.0 ┆ 1674.0 │
│ 21       ┆ 45175  ┆ 967.0  ┆ 1046.690647 ┆ 1554.0 ┆ 1789.0 │
│ 22       ┆ 26498  ┆ 967.0  ┆ 1063.059401 ┆ 1501.0 ┆ 1789.0 │
│ 23       ┆ 12906  ┆ 1026.0 ┆ 1178.4977

In [ ]:
hour_variability = (
    airport_hour
    .group_by("ADEP_mvt")
    .agg([
        pl.col("median").min().alias("min_hourly_median"),
        pl.col("median").max().alias("max_hourly_median"),

        (
            pl.col("median").max() -
            pl.col("median").min()
        ).alias("median_range"),

        pl.col("q95").min().alias("min_q95"),
        pl.col("q95").max().alias("max_q95"),

        (
            pl.col("q95").max() -
            pl.col("q95").min()
        ).alias("q95_range"),
    ])
    .sort("median_range", descending=True)
)

print(hour_variability)

shape: (10, 7)
┌──────────┬───────────────────┬───────────────────┬──────────────┬─────────┬─────────┬───────────┐
│ ADEP_mvt ┆ min_hourly_median ┆ max_hourly_median ┆ median_range ┆ min_q95 ┆ max_q95 ┆ q95_range │
│ ---      ┆ ---               ┆ ---               ┆ ---          ┆ ---     ┆ ---     ┆ ---       │
│ str      ┆ f64               ┆ f64               ┆ f64          ┆ f64     ┆ f64     ┆ f64       │
╞══════════╪═══════════════════╪═══════════════════╪══════════════╪═════════╪═════════╪═══════════╡
│ LSZH     ┆ 2.0               ┆ 870.0             ┆ 868.0        ┆ 8.0     ┆ 1436.0  ┆ 1428.0    │
│ EDDF     ┆ 356.5             ┆ 959.0             ┆ 602.5        ┆ 576.0   ┆ 1543.0  ┆ 967.0     │
│ EGLL     ┆ 911.0             ┆ 1493.0            ┆ 582.0        ┆ 1441.0  ┆ 2153.0  ┆ 712.0     │
│ LIRF     ┆ 897.0             ┆ 1429.0            ┆ 532.0        ┆ 1568.0  ┆ 5711.0  ┆ 4143.0    │
│ EDDM     ┆ 365.0             ┆ 841.0             ┆ 476.0        ┆ 430.0   ┆ 1612.0 

In [ ]:
weekday_stats = (
    local_df
    .group_by(["ADEP_mvt", "local_weekday"])
    .agg([
        pl.len().alias("n"),
        pl.col("TAXITIME_SEC_mvt").median().alias("median"),
        pl.col("TAXITIME_SEC_mvt").mean().alias("mean"),
        pl.col("TAXITIME_SEC_mvt").quantile(0.90).alias("q90"),
        pl.col("TAXITIME_SEC_mvt").quantile(0.95).alias("q95"),
    ])
    .sort(["ADEP_mvt", "local_weekday"])
)

print(weekday_stats)

shape: (70, 7)
┌──────────┬───────────────┬───────┬────────┬─────────────┬────────┬────────┐
│ ADEP_mvt ┆ local_weekday ┆ n     ┆ median ┆ mean        ┆ q90    ┆ q95    │
│ ---      ┆ ---           ┆ ---   ┆ ---    ┆ ---         ┆ ---    ┆ ---    │
│ str      ┆ i8            ┆ u32   ┆ f64    ┆ f64         ┆ f64    ┆ f64    │
╞══════════╪═══════════════╪═══════╪════════╪═════════════╪════════╪════════╡
│ EDDF     ┆ 1             ┆ 32462 ┆ 835.0  ┆ 864.43189   ┆ 1254.0 ┆ 1403.0 │
│ EDDF     ┆ 2             ┆ 31573 ┆ 824.0  ┆ 849.024863  ┆ 1229.0 ┆ 1369.0 │
│ EDDF     ┆ 3             ┆ 33474 ┆ 844.0  ┆ 873.608442  ┆ 1270.0 ┆ 1431.0 │
│ EDDF     ┆ 4             ┆ 32823 ┆ 830.0  ┆ 855.384304  ┆ 1242.0 ┆ 1384.0 │
│ EDDF     ┆ 5             ┆ 33928 ┆ 835.0  ┆ 856.770573  ┆ 1236.0 ┆ 1364.0 │
│ …        ┆ …             ┆ …     ┆ …      ┆ …           ┆ …      ┆ …      │
│ LTFM     ┆ 3             ┆ 39130 ┆ 966.0  ┆ 1066.726399 ┆ 1511.0 ┆ 1795.0 │
│ LTFM     ┆ 4             ┆ 38397 ┆ 966.0  ┆ 105

In [ ]:
hour_weekday = (
    local_df
    .group_by([
        "ADEP_mvt",
        "local_weekday",
        "local_hour"
    ])
    .agg([
        pl.len().alias("n"),
        pl.col("TAXITIME_SEC_mvt").median().alias("median"),
        pl.col("TAXITIME_SEC_mvt").mean().alias("mean"),
        pl.col("TAXITIME_SEC_mvt").quantile(0.90).alias("q90"),
        pl.col("TAXITIME_SEC_mvt").quantile(0.95).alias("q95"),
    ])
    .filter(pl.col("n") >= 100)
    .sort([
        "ADEP_mvt",
        "local_weekday",
        "local_hour"
    ])
)

print(hour_weekday)

shape: (1_386, 8)
┌──────────┬───────────────┬────────────┬──────┬────────┬─────────────┬────────┬────────┐
│ ADEP_mvt ┆ local_weekday ┆ local_hour ┆ n    ┆ median ┆ mean        ┆ q90    ┆ q95    │
│ ---      ┆ ---           ┆ ---        ┆ ---  ┆ ---    ┆ ---         ┆ ---    ┆ ---    │
│ str      ┆ i8            ┆ i8         ┆ u32  ┆ f64    ┆ f64         ┆ f64    ┆ f64    │
╞══════════╪═══════════════╪════════════╪══════╪════════╪═════════════╪════════╪════════╡
│ EDDF     ┆ 1             ┆ 4          ┆ 167  ┆ 955.0  ┆ 976.305389  ┆ 1223.0 ┆ 1317.0 │
│ EDDF     ┆ 1             ┆ 5          ┆ 172  ┆ 811.5  ┆ 827.0       ┆ 1020.0 ┆ 1123.0 │
│ EDDF     ┆ 1             ┆ 6          ┆ 916  ┆ 719.5  ┆ 750.453057  ┆ 1082.0 ┆ 1214.0 │
│ EDDF     ┆ 1             ┆ 7          ┆ 1513 ┆ 739.0  ┆ 772.664243  ┆ 1104.0 ┆ 1224.0 │
│ EDDF     ┆ 1             ┆ 8          ┆ 1775 ┆ 754.0  ┆ 778.822535  ┆ 1127.0 ┆ 1274.0 │
│ …        ┆ …             ┆ …          ┆ …    ┆ …      ┆ …           ┆ …      ┆ …

In [ ]:
extreme_local = (
    local_df
    .filter(
        (pl.col("TAXITIME_SEC_mvt") > 7200) |
        (pl.col("TAXITIME_SEC_mvt") < 0)
    )
    .sort("TAXITIME_SEC_mvt", descending=True)
)

print(extreme_local.head(50))

shape: (50, 10)
┌──────────┬───────────┬───────────┬───────────┬───┬───────────┬───────────┬───────────┬───────────┐
│ ADEP_mvt ┆ TAXITIME_ ┆ BLOCK_TIM ┆ local_dat ┆ … ┆ local_wee ┆ local_mon ┆ local_day ┆ is_weeken │
│ ---      ┆ SEC_mvt   ┆ E_UTC_mvt ┆ etime     ┆   ┆ kday      ┆ th        ┆ ---       ┆ d         │
│ str      ┆ ---       ┆ ---       ┆ ---       ┆   ┆ ---       ┆ ---       ┆ i16       ┆ ---       │
│          ┆ i32       ┆ datetime[ ┆ datetime[ ┆   ┆ i8        ┆ i8        ┆           ┆ bool      │
│          ┆           ┆ μs, UTC]  ┆ μs]       ┆   ┆           ┆           ┆           ┆           │
╞══════════╪═══════════╪═══════════╪═══════════╪═══╪═══════════╪═══════════╪═══════════╪═══════════╡
│ LIRF     ┆ 131167    ┆ 2025-02-1 ┆ 2025-02-1 ┆ … ┆ 1         ┆ 2         ┆ 41        ┆ false     │
│          ┆           ┆ 0         ┆ 0         ┆   ┆           ┆           ┆           ┆           │
│          ┆           ┆ 05:24:56  ┆ 06:24:56  ┆   ┆           ┆           

In [ ]:
local_df = (
    lf
    .with_columns(
        pl.col("BLOCK_TIME_UTC_mvt")
        .dt.convert_time_zone("UTC")
        .alias("utc_datetime")
    )
    .with_columns(
        pl.when(pl.col("ADEP_mvt") == "EDDF").then(
            pl.col("BLOCK_TIME_UTC_mvt").dt.convert_time_zone("Europe/Berlin").dt.replace_time_zone(None)
        )
        .when(pl.col("ADEP_mvt") == "EDDM").then(
            pl.col("BLOCK_TIME_UTC_mvt").dt.convert_time_zone("Europe/Berlin").dt.replace_time_zone(None)
        )
        .when(pl.col("ADEP_mvt") == "EGLL").then(
            pl.col("BLOCK_TIME_UTC_mvt").dt.convert_time_zone("Europe/London").dt.replace_time_zone(None)
        )
        .when(pl.col("ADEP_mvt") == "EHAM").then(
            pl.col("BLOCK_TIME_UTC_mvt").dt.convert_time_zone("Europe/Amsterdam").dt.replace_time_zone(None)
        )
        .when(pl.col("ADEP_mvt") == "LEBL").then(
            pl.col("BLOCK_TIME_UTC_mvt").dt.convert_time_zone("Europe/Madrid").dt.replace_time_zone(None)
        )
        .when(pl.col("ADEP_mvt") == "LEMD").then(
            pl.col("BLOCK_TIME_UTC_mvt").dt.convert_time_zone("Europe/Madrid").dt.replace_time_zone(None)
        )
        .when(pl.col("ADEP_mvt") == "LFPG").then(
            pl.col("BLOCK_TIME_UTC_mvt").dt.convert_time_zone("Europe/Paris").dt.replace_time_zone(None)
        )
        .when(pl.col("ADEP_mvt") == "LIRF").then(
            pl.col("BLOCK_TIME_UTC_mvt").dt.convert_time_zone("Europe/Rome").dt.replace_time_zone(None)
        )
        .when(pl.col("ADEP_mvt") == "LTFM").then(
            pl.col("BLOCK_TIME_UTC_mvt").dt.convert_time_zone("Europe/Istanbul").dt.replace_time_zone(None)
        )
        .when(pl.col("ADEP_mvt") == "LSZH").then(
            pl.col("BLOCK_TIME_UTC_mvt").dt.convert_time_zone("Europe/Zurich").dt.replace_time_zone(None)
        )
        .alias("local_datetime")
    )
    .with_columns([
        pl.col("local_datetime").dt.hour().alias("local_hour"),
        pl.col("local_datetime").dt.weekday().alias("local_weekday"),
        pl.col("local_datetime").dt.month().alias("local_month"),
        pl.col("local_datetime").dt.ordinal_day().alias("local_day"),
        (
            pl.col("local_datetime").dt.weekday() >= 6
        ).alias("is_weekend"),
    ])
    .select([
        "MVT_ID_mvt",
        "FLIGHT_mvt",
        "ADEP_mvt",
        "ADES_mvt",
        "BLOCK_TIME_UTC_mvt",
        "MVT_TIME_UTC_mvt",
        "TAXITIME_SEC_mvt",
        "local_datetime",
        "local_hour",
        "local_weekday",
        "local_month",
        "local_day",
        "is_weekend",
    ])
    .collect()
)

In [ ]:
(
    local_df
    .filter(
        (pl.col("TAXITIME_SEC_mvt") < 0) |
        (pl.col("TAXITIME_SEC_mvt") > 7200)
    )
    .with_columns(
        (
            pl.col("MVT_TIME_UTC_mvt") -
            pl.col("BLOCK_TIME_UTC_mvt")
        )
        .dt.total_seconds()
        .alias("reconstructed_taxi")
    )
    .select([
        "MVT_ID_mvt",
        "FLIGHT_mvt",
        "ADEP_mvt",
        "ADES_mvt",
        "BLOCK_TIME_UTC_mvt",
        "MVT_TIME_UTC_mvt",
        "TAXITIME_SEC_mvt",
        "reconstructed_taxi",
        "local_datetime",
        "local_hour",
    ])
    .sort("TAXITIME_SEC_mvt", descending=True)
    .head(100)
)

MVT_ID_mvt,FLIGHT_mvt,ADEP_mvt,ADES_mvt,BLOCK_TIME_UTC_mvt,MVT_TIME_UTC_mvt,TAXITIME_SEC_mvt,reconstructed_taxi,local_datetime,local_hour
f64,str,str,str,"datetime[μs, UTC]","datetime[μs, UTC]",i32,i64,datetime[μs],i8
1.83903219e8,"""NOSOS739""","""LIRF""","""LIMC""",2025-02-10 05:24:56 UTC,2025-02-11 17:51:03 UTC,131167,131167,2025-02-10 06:24:56,6
1.9019954e8,"""AMX071""","""LIRF""","""MMMX""",2025-06-24 15:29:54 UTC,2025-06-25 16:03:06 UTC,88392,88392,2025-06-24 17:29:54,17
1.92626268e8,"""CHHHH438""","""LIRF""","""ZGSZ""",2025-07-20 08:00:04 UTC,2025-07-21 08:28:56 UTC,88132,88132,2025-07-20 10:00:04,10
2.01198951e8,"""NOSOS610""","""LIRF""","""VRMM""",2025-12-27 18:14:57 UTC,2025-12-28 18:35:02 UTC,87605,87605,2025-12-27 19:14:57,19
2.00300302e8,"""BAWW561D""","""LIRF""","""EGLL""",2025-11-11 11:55:01 UTC,2025-11-12 12:14:59 UTC,87598,87598,2025-11-11 12:55:01,12
…,…,…,…,…,…,…,…,…,…
1.87033082e8,"""ETHTH574""","""LIRF""","""KORD""",2025-04-20 03:10:02 UTC,2025-04-20 07:57:58 UTC,17276,17276,2025-04-20 05:10:02,5
1.89113012e8,"""ISR342""","""LIRF""","""LLBG""",2025-05-09 17:19:56 UTC,2025-05-09 22:07:03 UTC,17227,17227,2025-05-09 19:19:56,19
1.92624067e8,"""ABR468""","""LIRF""","""LIMC""",2025-07-19 07:30:01 UTC,2025-07-19 12:16:58 UTC,17217,17217,2025-07-19 09:30:01,9


In [ ]:
target_bands = (
    local_df
    .with_columns(
        pl.when(pl.col("TAXITIME_SEC_mvt") < 0)
        .then(pl.lit("negative"))
        .when(pl.col("TAXITIME_SEC_mvt") <= 60)
        .then(pl.lit("0-60s"))
        .when(pl.col("TAXITIME_SEC_mvt") <= 300)
        .then(pl.lit("60-300s"))
        .when(pl.col("TAXITIME_SEC_mvt") <= 600)
        .then(pl.lit("300-600s"))
        .when(pl.col("TAXITIME_SEC_mvt") <= 900)
        .then(pl.lit("600-900s"))
        .when(pl.col("TAXITIME_SEC_mvt") <= 1200)
        .then(pl.lit("900-1200s"))
        .when(pl.col("TAXITIME_SEC_mvt") <= 1800)
        .then(pl.lit("1200-1800s"))
        .when(pl.col("TAXITIME_SEC_mvt") <= 3600)
        .then(pl.lit("1800-3600s"))
        .when(pl.col("TAXITIME_SEC_mvt") <= 7200)
        .then(pl.lit("3600-7200s"))
        .otherwise(pl.lit(">7200s"))
        .alias("target_band")
    )
    .group_by("target_band")
    .agg(pl.len().alias("n"))
)

print(target_bands)

shape: (10, 2)
┌─────────────┬────────┐
│ target_band ┆ n      │
│ ---         ┆ ---    │
│ str         ┆ u32    │
╞═════════════╪════════╡
│ 60-300s     ┆ 22204  │
│ 600-900s    ┆ 732296 │
│ >7200s      ┆ 584    │
│ 900-1200s   ┆ 605177 │
│ 1800-3600s  ┆ 70041  │
│ 3600-7200s  ┆ 3542   │
│ 300-600s    ┆ 237645 │
│ 0-60s       ┆ 748    │
│ negative    ┆ 369    │
│ 1200-1800s  ┆ 412441 │
└─────────────┴────────┘


In [ ]:
airport_bands = (
    local_df
    .with_columns(
        pl.when(pl.col("TAXITIME_SEC_mvt") < 0)
        .then(pl.lit("negative"))
        .when(pl.col("TAXITIME_SEC_mvt") <= 60)
        .then(pl.lit("0-60s"))
        .when(pl.col("TAXITIME_SEC_mvt") <= 300)
        .then(pl.lit("60-300s"))
        .when(pl.col("TAXITIME_SEC_mvt") <= 600)
        .then(pl.lit("300-600s"))
        .when(pl.col("TAXITIME_SEC_mvt") <= 900)
        .then(pl.lit("600-900s"))
        .when(pl.col("TAXITIME_SEC_mvt") <= 1200)
        .then(pl.lit("900-1200s"))
        .when(pl.col("TAXITIME_SEC_mvt") <= 1800)
        .then(pl.lit("1200-1800s"))
        .when(pl.col("TAXITIME_SEC_mvt") <= 3600)
        .then(pl.lit("1800-3600s"))
        .when(pl.col("TAXITIME_SEC_mvt") <= 7200)
        .then(pl.lit("3600-7200s"))
        .otherwise(pl.lit(">7200s"))
        .alias("target_band")
    )
    .group_by(["ADEP_mvt", "target_band"])
    .agg(pl.len().alias("n"))
    .sort(["ADEP_mvt", "target_band"])
)

print(airport_bands)

shape: (90, 3)
┌──────────┬─────────────┬───────┐
│ ADEP_mvt ┆ target_band ┆ n     │
│ ---      ┆ ---         ┆ ---   │
│ str      ┆ str         ┆ u32   │
╞══════════╪═════════════╪═══════╡
│ EDDF     ┆ 0-60s       ┆ 4     │
│ EDDF     ┆ 1200-1800s  ┆ 26737 │
│ EDDF     ┆ 1800-3600s  ┆ 2352  │
│ EDDF     ┆ 300-600s    ┆ 39602 │
│ EDDF     ┆ 3600-7200s  ┆ 73    │
│ …        ┆ …           ┆ …     │
│ LTFM     ┆ 60-300s     ┆ 224   │
│ LTFM     ┆ 600-900s    ┆ 95986 │
│ LTFM     ┆ 900-1200s   ┆ 94144 │
│ LTFM     ┆ >7200s      ┆ 21    │
│ LTFM     ┆ negative    ┆ 2     │
└──────────┴─────────────┴───────┘


## The Day Boundary Hypothesis

In [ ]:
day_artifacts = (
    local_df
    .with_columns(
        (
            pl.col("TAXITIME_SEC_mvt") / 3600
        ).alias("taxi_hours")
    )
    .filter(pl.col("taxi_hours") > 4)
    .with_columns(
        (
            pl.col("taxi_hours") -
            pl.col("taxi_hours").round()
        ).abs().alias("distance_from_integer_hour")
    )
    .sort("TAXITIME_SEC_mvt", descending=True)
)

print(day_artifacts.head(100))

shape: (100, 15)
┌────────────┬───────────┬──────────┬──────────┬───┬───────────┬───────────┬───────────┬───────────┐
│ MVT_ID_mvt ┆ FLIGHT_mv ┆ ADEP_mvt ┆ ADES_mvt ┆ … ┆ local_day ┆ is_weeken ┆ taxi_hour ┆ distance_ │
│ ---        ┆ t         ┆ ---      ┆ ---      ┆   ┆ ---       ┆ d         ┆ s         ┆ from_inte │
│ f64        ┆ ---       ┆ str      ┆ str      ┆   ┆ i16       ┆ ---       ┆ ---       ┆ ger_hour  │
│            ┆ str       ┆          ┆          ┆   ┆           ┆ bool      ┆ f64       ┆ ---       │
│            ┆           ┆          ┆          ┆   ┆           ┆           ┆           ┆ f64       │
╞════════════╪═══════════╪══════════╪══════════╪═══╪═══════════╪═══════════╪═══════════╪═══════════╡
│ 1.83903219 ┆ NOSOS739  ┆ LIRF     ┆ LIMC     ┆ … ┆ 41        ┆ false     ┆ 36.435278 ┆ 0.435278  │
│ e8         ┆           ┆          ┆          ┆   ┆           ┆           ┆           ┆           │
│ 1.9019954e ┆ AMX071    ┆ LIRF     ┆ MMMX     ┆ … ┆ 175       ┆ false    

In [ ]:
(
    local_df
    .filter(pl.col("TAXITIME_SEC_mvt") > 7200)
    .with_columns([
        (pl.col("TAXITIME_SEC_mvt") / 3600).alias("hours"),
        (pl.col("TAXITIME_SEC_mvt") / 86400).alias("days"),
    ])
    .select([
        "ADEP_mvt",
        "FLIGHT_mvt",
        "BLOCK_TIME_UTC_mvt",
        "MVT_TIME_UTC_mvt",
        "TAXITIME_SEC_mvt",
        "hours",
        "days",
    ])
    .sort("TAXITIME_SEC_mvt", descending=True)
    .head(100)
)

ADEP_mvt,FLIGHT_mvt,BLOCK_TIME_UTC_mvt,MVT_TIME_UTC_mvt,TAXITIME_SEC_mvt,hours,days
str,str,"datetime[μs, UTC]","datetime[μs, UTC]",i32,f64,f64
"""LIRF""","""NOSOS739""",2025-02-10 05:24:56 UTC,2025-02-11 17:51:03 UTC,131167,36.435278,1.518137
"""LIRF""","""AMX071""",2025-06-24 15:29:54 UTC,2025-06-25 16:03:06 UTC,88392,24.553333,1.023056
"""LIRF""","""CHHHH438""",2025-07-20 08:00:04 UTC,2025-07-21 08:28:56 UTC,88132,24.481111,1.020046
"""LIRF""","""NOSOS610""",2025-12-27 18:14:57 UTC,2025-12-28 18:35:02 UTC,87605,24.334722,1.013947
"""LIRF""","""BAWW561D""",2025-11-11 11:55:01 UTC,2025-11-12 12:14:59 UTC,87598,24.332778,1.013866
…,…,…,…,…,…,…
"""LIRF""","""ETHTH574""",2025-04-20 03:10:02 UTC,2025-04-20 07:57:58 UTC,17276,4.798889,0.199954
"""LIRF""","""ISR342""",2025-05-09 17:19:56 UTC,2025-05-09 22:07:03 UTC,17227,4.785278,0.199387
"""LIRF""","""ABR468""",2025-07-19 07:30:01 UTC,2025-07-19 12:16:58 UTC,17217,4.7825,0.199271


In [ ]:
df = df.with_columns(
    pl.when(pl.col("TAXITIME_SEC_mvt") < 0)
      .then(pl.lit("negative"))
      .when(pl.col("TAXITIME_SEC_mvt") > 7200)
      .then(pl.lit("extreme"))
      .when(pl.col("TAXITIME_SEC_mvt") > 3600)
      .then(pl.lit("very_long"))
      .otherwise(pl.lit("normal"))
      .alias("taxi_qc_class")
)

In [ ]:
print(df.schema)

missing = (
    df
    .select([
        pl.col(c).null_count().alias(c)
        for c in df.columns
    ])
)

print(missing)

Schema({'MVT_ID_mvt': Float64, 'FLIGHT_ID_mvt': Float64, 'FLIGHT_mvt': String, 'FLIGHT_RULE_mvt': String, 'ADEP_mvt': String, 'ADES_mvt': String, 'PHASE_mvt': String, 'MVT_TIME_UTC_mvt': Datetime(time_unit='us', time_zone='UTC'), 'BLOCK_TIME_UTC_mvt': Datetime(time_unit='us', time_zone='UTC'), 'SCHED_TIME_UTC_mvt': Datetime(time_unit='us', time_zone='UTC'), 'AIRCRAFT_TYPE_mvt': String, 'RUNWAY_mvt': String, 'STAND_mvt': String, 'TAXITIME_SEC_mvt': Int32, 'taxi_qc_class': String})
shape: (1, 15)
┌───────────┬───────────┬───────────┬───────────┬───┬───────────┬───────────┬───────────┬──────────┐
│ MVT_ID_mv ┆ FLIGHT_ID ┆ FLIGHT_mv ┆ FLIGHT_RU ┆ … ┆ RUNWAY_mv ┆ STAND_mvt ┆ TAXITIME_ ┆ taxi_qc_ │
│ t         ┆ _mvt      ┆ t         ┆ LE_mvt    ┆   ┆ t         ┆ ---       ┆ SEC_mvt   ┆ class    │
│ ---       ┆ ---       ┆ ---       ┆ ---       ┆   ┆ ---       ┆ u32       ┆ ---       ┆ ---      │
│ u32       ┆ u32       ┆ u32       ┆ u32       ┆   ┆ u32       ┆           ┆ u32       ┆ u32   

In [ ]:
print(df.schema)

missing = (
    df
    .select([
        pl.col(c).null_count().alias(c)
        for c in df.columns
    ])
)

print(missing)

Schema({'MVT_ID_mvt': Float64, 'FLIGHT_ID_mvt': Float64, 'FLIGHT_mvt': String, 'FLIGHT_RULE_mvt': String, 'ADEP_mvt': String, 'ADES_mvt': String, 'PHASE_mvt': String, 'MVT_TIME_UTC_mvt': Datetime(time_unit='us', time_zone='UTC'), 'BLOCK_TIME_UTC_mvt': Datetime(time_unit='us', time_zone='UTC'), 'SCHED_TIME_UTC_mvt': Datetime(time_unit='us', time_zone='UTC'), 'AIRCRAFT_TYPE_mvt': String, 'RUNWAY_mvt': String, 'STAND_mvt': String, 'TAXITIME_SEC_mvt': Int32, 'taxi_qc_class': String})
shape: (1, 15)
┌───────────┬───────────┬───────────┬───────────┬───┬───────────┬───────────┬───────────┬──────────┐
│ MVT_ID_mv ┆ FLIGHT_ID ┆ FLIGHT_mv ┆ FLIGHT_RU ┆ … ┆ RUNWAY_mv ┆ STAND_mvt ┆ TAXITIME_ ┆ taxi_qc_ │
│ t         ┆ _mvt      ┆ t         ┆ LE_mvt    ┆   ┆ t         ┆ ---       ┆ SEC_mvt   ┆ class    │
│ ---       ┆ ---       ┆ ---       ┆ ---       ┆   ┆ ---       ┆ u32       ┆ ---       ┆ ---      │
│ u32       ┆ u32       ┆ u32       ┆ u32       ┆   ┆ u32       ┆           ┆ u32       ┆ u32   

In [ ]:
categorical_cols = [
    "FLIGHT_RULE_mvt",
    "ADEP_mvt",
    "ADES_mvt",
    "PHASE_mvt",
    "AIRCRAFT_TYPE_mvt",
    "RUNWAY_mvt",
    "STAND_mvt",
]

for c in categorical_cols:
    print(f"\n===== {c} =====")
    print(
        df
        .group_by(c)
        .agg(pl.len().alias("n"))
        .sort("n", descending=True)
        .head(30)
    )


===== FLIGHT_RULE_mvt =====
shape: (3, 2)
┌─────────────────┬─────────┐
│ FLIGHT_RULE_mvt ┆ n       │
│ ---             ┆ ---     │
│ str             ┆ u32     │
╞═════════════════╪═════════╡
│ I               ┆ 2081914 │
│ V               ┆ 2930    │
│ null            ┆ 203     │
└─────────────────┴─────────┘

===== ADEP_mvt =====
shape: (10, 2)
┌──────────┬────────┐
│ ADEP_mvt ┆ n      │
│ ---      ┆ ---    │
│ str      ┆ u32    │
╞══════════╪════════╡
│ LTFM     ┆ 272965 │
│ EHAM     ┆ 247951 │
│ LFPG     ┆ 239552 │
│ EGLL     ┆ 239546 │
│ EDDF     ┆ 230141 │
│ LEMD     ┆ 212242 │
│ LEBL     ┆ 179705 │
│ EDDM     ┆ 167334 │
│ LIRF     ┆ 160704 │
│ LSZH     ┆ 134907 │
└──────────┴────────┘

===== ADES_mvt =====
shape: (30, 2)
┌──────────┬───────┐
│ ADES_mvt ┆ n     │
│ ---      ┆ ---   │
│ str      ┆ u32   │
╞══════════╪═══════╡
│ EGLL     ┆ 39269 │
│ EHAM     ┆ 37273 │
│ EDDF     ┆ 35239 │
│ LFPG     ┆ 34585 │
│ LEMD     ┆ 34356 │
│ …        ┆ …     │
│ LFMN     ┆ 18229 │
│ EGCC   

In [ ]:
NULL_LABELS = ["", "NA", "N/A", "UNKNOWN", "Unknown", "unknown"]

df = df.with_columns([
    pl.when(
        pl.col(c)
        .cast(pl.String)
        .str.strip_chars()
        .is_in(NULL_LABELS)
    )
    .then(None)
    .otherwise(
        pl.col(c).cast(pl.String).str.strip_chars()
    )
    .alias(c)
    for c in categorical_cols
])

In [ ]:
print(
    df.group_by("PHASE_mvt")
      .agg(pl.len().alias("n"))
      .sort("n", descending=True)
)

shape: (1, 2)
┌───────────┬─────────┐
│ PHASE_mvt ┆ n       │
│ ---       ┆ ---     │
│ str       ┆ u32     │
╞═══════════╪═════════╡
│ DEP       ┆ 2085047 │
└───────────┴─────────┘


In [ ]:
print(
    df.select(
        pl.col("MVT_ID_mvt").n_unique().alias("unique"),
        pl.len().alias("rows")
    )
)

shape: (1, 2)
┌─────────┬─────────┐
│ unique  ┆ rows    │
│ ---     ┆ ---     │
│ u32     ┆ u32     │
╞═════════╪═════════╡
│ 2085047 ┆ 2085047 │
└─────────┴─────────┘


In [ ]:
duplicates = (
    df
    .group_by("MVT_ID_mvt")
    .agg(pl.len().alias("n"))
    .filter(pl.col("n") > 1)
    .sort("n", descending=True)
)

print(duplicates.head(50))

shape: (0, 2)
┌────────────┬─────┐
│ MVT_ID_mvt ┆ n   │
│ ---        ┆ --- │
│ f64        ┆ u32 │
╞════════════╪═════╡
└────────────┴─────┘


In [ ]:
for c in ["MVT_ID_mvt", "FLIGHT_ID_mvt"]:
    print(
        c,
        df.select([
            pl.len().alias("n"),
            pl.col(c).n_unique().alias("unique"),
            pl.col(c).null_count().alias("null")
        ])
    )

MVT_ID_mvt shape: (1, 3)
┌─────────┬─────────┬──────┐
│ n       ┆ unique  ┆ null │
│ ---     ┆ ---     ┆ ---  │
│ u32     ┆ u32     ┆ u32  │
╞═════════╪═════════╪══════╡
│ 2085047 ┆ 2085047 ┆ 0    │
└─────────┴─────────┴──────┘
FLIGHT_ID_mvt shape: (1, 3)
┌─────────┬─────────┬───────┐
│ n       ┆ unique  ┆ null  │
│ ---     ┆ ---     ┆ ---   │
│ u32     ┆ u32     ┆ u32   │
╞═════════╪═════════╪═══════╡
│ 2085047 ┆ 2062621 ┆ 22424 │
└─────────┴─────────┴───────┘


In [ ]:
df = df.with_columns(
    (
        (
            pl.col("MVT_TIME_UTC_mvt")
            - pl.col("BLOCK_TIME_UTC_mvt")
        )
        .dt.total_seconds()
        .cast(pl.Int64)
    ).alias("reconstructed_taxi")
)

In [ ]:
print(
    df.select(
        pl.col("TAXITIME_SEC_mvt").alias("reported"),
        pl.col("reconstructed_taxi")
    )
    .with_columns(
        (
            pl.col("reported") -
            pl.col("reconstructed_taxi")
        ).alias("difference")
    )
    .select([
        pl.col("difference").abs().max().alias("max_abs_difference"),
        (pl.col("difference") != 0).sum().alias("mismatches")
    ])
)

shape: (1, 2)
┌────────────────────┬────────────┐
│ max_abs_difference ┆ mismatches │
│ ---                ┆ ---        │
│ i64                ┆ u32        │
╞════════════════════╪════════════╡
│ 0                  ┆ 0          │
└────────────────────┴────────────┘


In [ ]:
df = df.with_columns(
    (pl.col("TAXITIME_SEC_mvt") < 0)
    .alias("qc_negative_taxi")
)

In [ ]:
df = df.with_columns([
    (
        pl.col("TAXITIME_SEC_mvt") > 3600
    ).alias("qc_taxi_gt_1h"),

    (
        pl.col("TAXITIME_SEC_mvt") > 7200
    ).alias("qc_taxi_gt_2h"),

    (
        pl.col("TAXITIME_SEC_mvt") > 86400
    ).alias("qc_taxi_gt_24h"),
])

In [ ]:
print(
    df.select([
        pl.col("qc_negative_taxi").sum().alias("negative"),
        pl.col("qc_taxi_gt_1h").sum().alias(">1h"),
        pl.col("qc_taxi_gt_2h").sum().alias(">2h"),
        pl.col("qc_taxi_gt_24h").sum().alias(">24h"),
    ])
)

shape: (1, 4)
┌──────────┬──────┬─────┬──────┐
│ negative ┆ >1h  ┆ >2h ┆ >24h │
│ ---      ┆ ---  ┆ --- ┆ ---  │
│ u32      ┆ u32  ┆ u32 ┆ u32  │
╞══════════╪══════╪═════╪══════╡
│ 369      ┆ 4126 ┆ 584 ┆ 15   │
└──────────┴──────┴─────┴──────┘


In [ ]:
df = df.with_columns(
    pl.when(pl.col("TAXITIME_SEC_mvt") < 0)
      .then(pl.lit("negative"))
      .when(pl.col("TAXITIME_SEC_mvt") > 86400)
      .then(pl.lit("day_boundary"))
      .when(pl.col("TAXITIME_SEC_mvt") > 7200)
      .then(pl.lit("extreme"))
      .when(pl.col("TAXITIME_SEC_mvt") > 3600)
      .then(pl.lit("very_long"))
      .otherwise(pl.lit("valid"))
      .alias("qc_class")
)

In [ ]:
print(
    df.group_by("qc_class")
      .agg(pl.len().alias("n"))
      .sort("n", descending=True)
)

shape: (5, 2)
┌──────────────┬─────────┐
│ qc_class     ┆ n       │
│ ---          ┆ ---     │
│ str          ┆ u32     │
╞══════════════╪═════════╡
│ valid        ┆ 2080552 │
│ very_long    ┆ 3542    │
│ extreme      ┆ 569     │
│ negative     ┆ 369     │
│ day_boundary ┆ 15      │
└──────────────┴─────────┘


In [ ]:
timestamp_cols = [
    "MVT_TIME_UTC_mvt",
    "BLOCK_TIME_UTC_mvt",
    "SCHED_TIME_UTC_mvt",
]

print(
    df.select([
        pl.col(c).null_count().alias(c)
        for c in timestamp_cols
    ])
)

shape: (1, 3)
┌──────────────────┬────────────────────┬────────────────────┐
│ MVT_TIME_UTC_mvt ┆ BLOCK_TIME_UTC_mvt ┆ SCHED_TIME_UTC_mvt │
│ ---              ┆ ---                ┆ ---                │
│ u32              ┆ u32                ┆ u32                │
╞══════════════════╪════════════════════╪════════════════════╡
│ 0                ┆ 0                  ┆ 0                  │
└──────────────────┴────────────────────┴────────────────────┘


In [ ]:
print(
    df.filter(
        pl.col("MVT_TIME_UTC_mvt") <
        pl.col("BLOCK_TIME_UTC_mvt")
    ).select([
        "MVT_ID_mvt",
        "FLIGHT_mvt",
        "ADEP_mvt",
        "BLOCK_TIME_UTC_mvt",
        "MVT_TIME_UTC_mvt",
        "TAXITIME_SEC_mvt"
    ]).head(20)
)

shape: (20, 6)
┌──────────────┬────────────┬──────────┬────────────────────┬───────────────────┬──────────────────┐
│ MVT_ID_mvt   ┆ FLIGHT_mvt ┆ ADEP_mvt ┆ BLOCK_TIME_UTC_mvt ┆ MVT_TIME_UTC_mvt  ┆ TAXITIME_SEC_mvt │
│ ---          ┆ ---        ┆ ---      ┆ ---                ┆ ---               ┆ ---              │
│ f64          ┆ str        ┆ str      ┆ datetime[μs, UTC]  ┆ datetime[μs, UTC] ┆ i32              │
╞══════════════╪════════════╪══════════╪════════════════════╪═══════════════════╪══════════════════╡
│ 1.82977114e8 ┆ IBE2601    ┆ LEBL     ┆ 2025-01-15         ┆ 2025-01-15        ┆ -8               │
│              ┆            ┆          ┆ 00:55:05 UTC       ┆ 00:54:57 UTC      ┆                  │
│ 1.82379342e8 ┆ AFR15WK    ┆ LFPG     ┆ 2025-01-03         ┆ 2025-01-03        ┆ -11              │
│              ┆            ┆          ┆ 20:28:05 UTC       ┆ 20:27:54 UTC      ┆                  │
│ 1.82363682e8 ┆ CAO3158    ┆ LFPG     ┆ 2025-01-14         ┆ 2025-01-14    

In [ ]:
df = df.with_columns(
    (
        (
            pl.col("BLOCK_TIME_UTC_mvt")
            - pl.col("SCHED_TIME_UTC_mvt")
        )
        .dt.total_seconds()
    ).alias("departure_delay_sec")
)

In [ ]:
print(
    df.select([
        pl.col("departure_delay_sec").quantile(0.001).alias("q001"),
        pl.col("departure_delay_sec").quantile(0.01).alias("q01"),
        pl.col("departure_delay_sec").quantile(0.5).alias("q50"),
        pl.col("departure_delay_sec").quantile(0.9).alias("q90"),
        pl.col("departure_delay_sec").quantile(0.99).alias("q99"),
        pl.col("departure_delay_sec").quantile(0.999).alias("q999")
    ])
)

shape: (1, 6)
┌─────────┬────────┬───────┬────────┬────────┬─────────┐
│ q001    ┆ q01    ┆ q50   ┆ q90    ┆ q99    ┆ q999    │
│ ---     ┆ ---    ┆ ---   ┆ ---    ┆ ---    ┆ ---     │
│ f64     ┆ f64    ┆ f64   ┆ f64    ┆ f64    ┆ f64     │
╞═════════╪════════╪═══════╪════════╪════════╪═════════╡
│ -1805.0 ┆ -761.0 ┆ 415.0 ┆ 2396.0 ┆ 8163.0 ┆ 22084.0 │
└─────────┴────────┴───────┴────────┴────────┴─────────┘


In [ ]:
airport_tz = {
    "EDDF": "Europe/Berlin",
    "EDDM": "Europe/Berlin",
    "EGLL": "Europe/London",
    "EHAM": "Europe/Amsterdam",
    "LEBL": "Europe/Madrid",
    "LEMD": "Europe/Madrid",
    "LFPG": "Europe/Paris",
    "LIRF": "Europe/Rome",
    "LTFM": "Europe/Istanbul",
    "LSZH": "Europe/Zurich",
}

## Primary Preprocessing

In [ ]:
TARGET = "TAXITIME_SEC_mvt"

QC_DROP = {
    "negative",
    "extreme",
    "day_boundary",
}


df = df.with_columns(
    pl.col("MVT_ID_mvt").cast(pl.UInt64),
    pl.col("FLIGHT_ID_mvt").cast(pl.UInt64),
    pl.col(TARGET).cast(pl.Int32),
)




required_cols = [
    "MVT_ID_mvt",
    "FLIGHT_ID_mvt",
    "FLIGHT_mvt",
    "ADEP_mvt",
    "ADES_mvt",
    "PHASE_mvt",
    "MVT_TIME_UTC_mvt",
    "BLOCK_TIME_UTC_mvt",
    TARGET,
]

df = df.filter(
    pl.all_horizontal(
        pl.col(c).is_not_null()
        for c in required_cols
    )
)


df = df.with_columns(
    pl.when(pl.col(TARGET) < 0)
    .then(pl.lit("negative"))
    .when(pl.col(TARGET) > 86400)
    .then(pl.lit("day_boundary"))
    .when(pl.col(TARGET) > 7200)
    .then(pl.lit("extreme"))
    .when(pl.col(TARGET) > 3600)
    .then(pl.lit("very_long"))
    .otherwise(pl.lit("valid"))
    .alias("taxi_qc_class")
)




df_clean = df.filter(
    (pl.col(TARGET) > 0) &
    (pl.col(TARGET) <= 7200)
)



df_strict = df.filter(
    (pl.col(TARGET) > 0) &
    (pl.col(TARGET) <= 3600)
)

In [ ]:
categorical_cols = [
    "FLIGHT_RULE_mvt",
    "ADEP_mvt",
    "ADES_mvt",
    "AIRCRAFT_TYPE_mvt",
    "RUNWAY_mvt",
    "STAND_mvt",
]

df_clean = df_clean.with_columns(
    [
        pl.col(c).fill_null("__MISSING__").cast(pl.String)
        for c in categorical_cols
    ]
)

In [ ]:
DROP_COLS = [
    TARGET,
    "taxi_qc_class",

    # identifiers
    "MVT_ID_mvt",
    "FLIGHT_ID_mvt",
    "FLIGHT_mvt",

    # redundant / constant
    "PHASE_mvt",

    # raw timestamps after temporal extraction
    "MVT_TIME_UTC_mvt",
    "BLOCK_TIME_UTC_mvt",
    "SCHED_TIME_UTC_mvt",
]

In [ ]:
df_clean = df_clean.with_columns(
    (
        pl.col("BLOCK_TIME_UTC_mvt")
        - pl.col("MVT_TIME_UTC_mvt")
    )
    .dt.total_seconds()
    .alias("reconstructed_target")
)

In [ ]:
model_df = df_clean.drop(
    [c for c in DROP_COLS if c in df_clean.columns]
)

## Validating the targets

In [ ]:
print("RAW:", df.height)
print("CLEAN:", df_clean.height)
print("STRICT:", df_strict.height)

print(
    df_clean
    .select(
        pl.len().alias("n"),
        pl.col(TARGET).min().alias("min"),
        pl.col(TARGET).quantile(0.01).alias("q01"),
        pl.col(TARGET).median().alias("median"),
        pl.col(TARGET).mean().alias("mean"),
        pl.col(TARGET).quantile(0.90).alias("q90"),
        pl.col(TARGET).quantile(0.95).alias("q95"),
        pl.col(TARGET).quantile(0.99).alias("q99"),
        pl.col(TARGET).max().alias("max"),
        pl.col(TARGET).std().alias("std"),
    )
)

RAW: 2062420
CLEAN: 2062180
STRICT: 2059251
shape: (1, 10)
┌─────────┬─────┬───────┬────────┬───┬────────┬────────┬──────┬────────────┐
│ n       ┆ min ┆ q01   ┆ median ┆ … ┆ q95    ┆ q99    ┆ max  ┆ std        │
│ ---     ┆ --- ┆ ---   ┆ ---    ┆   ┆ ---    ┆ ---    ┆ ---  ┆ ---        │
│ u32     ┆ i32 ┆ f64   ┆ f64    ┆   ┆ f64    ┆ f64    ┆ i32  ┆ f64        │
╞═════════╪═════╪═══════╪════════╪═══╪════════╪════════╪══════╪════════════╡
│ 2062180 ┆ 1   ┆ 299.0 ┆ 911.0  ┆ … ┆ 1682.0 ┆ 2291.0 ┆ 7195 ┆ 408.957299 │
└─────────┴─────┴───────┴────────┴───┴────────┴────────┴──────┴────────────┘


In [ ]:
print(
    df_clean
    .group_by("ADEP_mvt")
    .agg(
        pl.len().alias("n"),
        pl.col(TARGET).median().alias("median"),
        pl.col(TARGET).mean().alias("mean"),
        pl.col(TARGET).quantile(0.90).alias("q90"),
        pl.col(TARGET).quantile(0.95).alias("q95"),
        pl.col(TARGET).quantile(0.99).alias("q99"),
        pl.col(TARGET).max().alias("max"),
    )
    .sort("median")
)

shape: (10, 8)
┌──────────┬────────┬────────┬─────────────┬────────┬────────┬────────┬──────┐
│ ADEP_mvt ┆ n      ┆ median ┆ mean        ┆ q90    ┆ q95    ┆ q99    ┆ max  │
│ ---      ┆ ---    ┆ ---    ┆ ---         ┆ ---    ┆ ---    ┆ ---    ┆ ---  │
│ str      ┆ u32    ┆ f64    ┆ f64         ┆ f64    ┆ f64    ┆ f64    ┆ i32  │
╞══════════╪════════╪════════╪═════════════╪════════╪════════╪════════╪══════╡
│ LSZH     ┆ 132538 ┆ 712.0  ┆ 741.853287  ┆ 1080.0 ┆ 1240.0 ┆ 1705.0 ┆ 6878 │
│ EHAM     ┆ 243863 ┆ 744.0  ┆ 787.579707  ┆ 1158.0 ┆ 1308.0 ┆ 1766.0 ┆ 6711 │
│ EDDM     ┆ 166325 ┆ 774.0  ┆ 810.552581  ┆ 1134.0 ┆ 1318.0 ┆ 1916.0 ┆ 6850 │
│ EDDF     ┆ 228136 ┆ 836.0  ┆ 862.543386  ┆ 1249.0 ┆ 1397.0 ┆ 1816.0 ┆ 6566 │
│ LEBL     ┆ 177844 ┆ 905.0  ┆ 955.661602  ┆ 1355.0 ┆ 1499.0 ┆ 1957.0 ┆ 3604 │
│ LFPG     ┆ 235686 ┆ 954.0  ┆ 1016.082372 ┆ 1443.0 ┆ 1679.0 ┆ 2399.0 ┆ 7085 │
│ LTFM     ┆ 269328 ┆ 963.0  ┆ 1049.546026 ┆ 1501.0 ┆ 1747.0 ┆ 2532.0 ┆ 7129 │
│ LEMD     ┆ 211303 ┆ 984.0  ┆ 1014.2

## Data Split

In [ ]:
local_frames = []

for airport, tz in airport_tz.items():
    part = (
        df_clean
        .filter(pl.col("ADEP_mvt") == airport)
        .with_columns(
            pl.col("BLOCK_TIME_UTC_mvt")
            .dt.convert_time_zone(tz)
            .dt.replace_time_zone(None) # Remove timezone for concatenation compatibility
            .alias("local_datetime")
        )
        .with_columns(
            pl.col("local_datetime").dt.hour().cast(pl.Int8).alias("local_hour"),
            pl.col("local_datetime").dt.weekday().cast(pl.Int8).alias("local_weekday"),
            pl.col("local_datetime").dt.month().cast(pl.Int8).alias("local_month"),
            pl.col("local_datetime").dt.day().cast(pl.Int16).alias("local_day"),
        )
        .with_columns(
            (pl.col("local_weekday") >= 6).alias("is_weekend")
        )
    )

    local_frames.append(part)

df_clean = pl.concat(local_frames)

In [ ]:
print(df_clean.schema)

print(
    df_clean
    .select(
        pl.col("local_datetime").min().alias("min_local"),
        pl.col("local_datetime").max().alias("max_local")
    )
)

df_clean = df_clean.sort("local_datetime")

print(
    df_clean
    .group_by_dynamic(
        "local_datetime",
        every="1mo",
    )
    .agg(
        pl.len().alias("n"),
        pl.col("TAXITIME_SEC_mvt").median().alias("median"),
        pl.col("TAXITIME_SEC_mvt").mean().alias("mean"),
    )
    .sort("local_datetime")
)

Schema({'MVT_ID_mvt': UInt64, 'FLIGHT_ID_mvt': UInt64, 'FLIGHT_mvt': String, 'FLIGHT_RULE_mvt': String, 'ADEP_mvt': String, 'ADES_mvt': String, 'PHASE_mvt': String, 'MVT_TIME_UTC_mvt': Datetime(time_unit='us', time_zone='UTC'), 'BLOCK_TIME_UTC_mvt': Datetime(time_unit='us', time_zone='UTC'), 'SCHED_TIME_UTC_mvt': Datetime(time_unit='us', time_zone='UTC'), 'AIRCRAFT_TYPE_mvt': String, 'RUNWAY_mvt': String, 'STAND_mvt': String, 'TAXITIME_SEC_mvt': Int32, 'taxi_qc_class': String, 'reconstructed_taxi': Int64, 'qc_negative_taxi': Boolean, 'qc_taxi_gt_1h': Boolean, 'qc_taxi_gt_2h': Boolean, 'qc_taxi_gt_24h': Boolean, 'qc_class': String, 'departure_delay_sec': Int64, 'reconstructed_target': Int64, 'local_datetime': Datetime(time_unit='us', time_zone=None), 'local_hour': Int8, 'local_weekday': Int8, 'local_month': Int8, 'local_day': Int16, 'is_weekend': Boolean})
shape: (1, 2)
┌─────────────────────┬─────────────────────┐
│ min_local           ┆ max_local           │
│ ---                 ┆ --

In [ ]:
df_clean = df_clean.sort("local_datetime")

train = df_clean.filter(
    pl.col("local_datetime") < pl.datetime(2025, 10, 1)
)

valid = df_clean.filter(
    (pl.col("local_datetime") >= pl.datetime(2025, 10, 1)) &
    (pl.col("local_datetime") < pl.datetime(2025, 12, 1))
)

test = df_clean.filter(
    pl.col("local_datetime") >= pl.datetime(2025, 12, 1)
)

print("TRAIN:", train.shape)
print("VALID:", valid.shape)
print("TEST:", test.shape)

print(
    train.select(
        pl.col("local_datetime").min().alias("train_min"),
        pl.col("local_datetime").max().alias("train_max"),
    )
)

print(
    valid.select(
        pl.col("local_datetime").min().alias("valid_min"),
        pl.col("local_datetime").max().alias("valid_max"),
    )
)

print(
    test.select(
        pl.col("local_datetime").min().alias("test_min"),
        pl.col("local_datetime").max().alias("test_max"),
    )
)

TRAIN: (1553484, 29)
VALID: (344686, 29)
TEST: (164010, 29)
shape: (1, 2)
┌─────────────────────┬─────────────────────┐
│ train_min           ┆ train_max           │
│ ---                 ┆ ---                 │
│ datetime[μs]        ┆ datetime[μs]        │
╞═════════════════════╪═════════════════════╡
│ 2025-01-01 01:21:55 ┆ 2025-09-30 23:58:55 │
└─────────────────────┴─────────────────────┘
shape: (1, 2)
┌─────────────────────┬─────────────────────┐
│ valid_min           ┆ valid_max           │
│ ---                 ┆ ---                 │
│ datetime[μs]        ┆ datetime[μs]        │
╞═════════════════════╪═════════════════════╡
│ 2025-10-01 00:01:03 ┆ 2025-11-30 23:59:03 │
└─────────────────────┴─────────────────────┘
shape: (1, 2)
┌─────────────────────┬─────────────────────┐
│ test_min            ┆ test_max            │
│ ---                 ┆ ---                 │
│ datetime[μs]        ┆ datetime[μs]        │
╞═════════════════════╪═════════════════════╡
│ 2025-12-01 00:02:56 ┆ 

## Feature Engineering

In [ ]:
def feature_engineer(df: pl.DataFrame) -> pl.DataFrame:
    return (
        df
        .with_columns([
            # Cyclical time
            (2 * np.pi * pl.col("local_hour") / 24)
                .sin()
                .alias("hour_sin"),

            (2 * np.pi * pl.col("local_hour") / 24)
                .cos()
                .alias("hour_cos"),

            (2 * np.pi * pl.col("local_weekday") / 7)
                .sin()
                .alias("weekday_sin"),

            (2 * np.pi * pl.col("local_weekday") / 7)
                .cos()
                .alias("weekday_cos"),

            (2 * np.pi * (pl.col("local_month") - 1) / 12)
                .sin()
                .alias("month_sin"),

            (2 * np.pi * (pl.col("local_month") - 1) / 12)
                .cos()
                .alias("month_cos"),

            # Time
            (
                pl.col("local_hour") * 60
                + pl.col("local_datetime").dt.minute()
            ).alias("time_of_day_min"),

            (
                pl.col("local_hour") * 3600
                + pl.col("local_datetime").dt.minute() * 60
                + pl.col("local_datetime").dt.second()
            ).alias("seconds_since_midnight"),

            # Calendar
            pl.col("local_datetime").dt.week()
                .alias("local_week"),

            pl.col("local_datetime").dt.ordinal_day()
                .alias("local_day_of_year"),

            # Missingness / availability
            pl.col("SCHED_TIME_UTC_mvt")
                .is_not_null()
                .alias("has_schedule"),

            pl.col("RUNWAY_mvt")
                .is_not_null()
                .alias("has_runway"),

            pl.col("STAND_mvt")
                .is_not_null()
                .alias("has_stand"),

            # Route
            pl.concat_str(
                ["ADEP_mvt", "ADES_mvt"],
                separator="→"
            ).alias("route"),

            # Airport × hour
            pl.concat_str(
                [
                    "ADEP_mvt",
                    pl.col("local_hour").cast(pl.String)
                ],
                separator="_"
            ).alias("airport_hour"),

            # Airport × weekday
            pl.concat_str(
                [
                    "ADEP_mvt",
                    pl.col("local_weekday").cast(pl.String)
                ],
                separator="_"
            ).alias("airport_weekday"),

            # Aircraft × airport
            pl.concat_str(
                [
                    "AIRCRAFT_TYPE_mvt",
                    "ADEP_mvt"
                ],
                separator="_"
            ).alias("aircraft_airport"),

            # Runway × airport
            pl.concat_str(
                [
                    "ADEP_mvt",
                    "RUNWAY_mvt"
                ],
                separator="_"
            ).alias("runway_airport"),
        ])
    )

In [ ]:
train_fe = feature_engineer(train)
valid_fe = feature_engineer(valid)
test_fe = feature_engineer(test)

In [ ]:
def encode_features(train, valid, test):
    ordinal_cols = [
        "FLIGHT_mvt",
        "PHASE_mvt",
        "FLIGHT_RULE_mvt",
        "ADEP_mvt",
        "ADES_mvt",
        "AIRCRAFT_TYPE_mvt",
        "RUNWAY_mvt",
        "STAND_mvt",
        "taxi_qc_class",
        "qc_class",
        "route",
        "airport_hour",
        "airport_weekday",
        "aircraft_airport",
        "runway_airport",
    ]

    binary_cols = [
        "qc_negative_taxi",
        "qc_taxi_gt_1h",
        "qc_taxi_gt_2h",
        "qc_taxi_gt_24h",
        "has_schedule",
        "has_runway",
        "has_stand",
        "is_weekend",
    ]

    # Fit ordinal mappings using TRAIN only
    mappings = {}

    for col in ordinal_cols:
        if col not in train.columns:
            continue

        categories = (
            train
            .select(pl.col(col).drop_nulls().unique().sort())
            .to_series()
            .to_list()
        )

        mappings[col] = {
            value: idx
            for idx, value in enumerate(categories, start=1)
        }

    def transform(df):
        exprs = []

        for col, mapping in mappings.items():
            exprs.append(
                pl.col(col)
                .replace(mapping, default=0)
                .cast(pl.Int32)
                .alias(col)
            )

        for col in binary_cols:
            if col in df.columns:
                exprs.append(
                    pl.col(col)
                    .fill_null(False)
                    .cast(pl.Int8)
                    .alias(col)
                )

        return df.with_columns(exprs)

    train_enc = transform(train)
    valid_enc = transform(valid)
    test_enc = transform(test)

    return train_enc, valid_enc, test_enc, mappings

In [ ]:
train_enc, valid_enc, test_enc, encoders = encode_features(
    train_fe,
    valid_fe,
    test_fe,
)

/tmp/ipykernel_7130/795360029.py:56: DeprecationWarning: the `default` parameter for `replace` is deprecated. Use `replace_strict` instead to set a default while replacing values.
(Deprecated in version 1.0.0)
  .replace(mapping, default=0)


In [ ]:
train_enc

MVT_ID_mvt,FLIGHT_ID_mvt,FLIGHT_mvt,FLIGHT_RULE_mvt,ADEP_mvt,ADES_mvt,PHASE_mvt,MVT_TIME_UTC_mvt,BLOCK_TIME_UTC_mvt,SCHED_TIME_UTC_mvt,AIRCRAFT_TYPE_mvt,RUNWAY_mvt,STAND_mvt,TAXITIME_SEC_mvt,taxi_qc_class,reconstructed_taxi,qc_negative_taxi,qc_taxi_gt_1h,qc_taxi_gt_2h,qc_taxi_gt_24h,qc_class,departure_delay_sec,reconstructed_target,local_datetime,local_hour,local_weekday,local_month,local_day,is_weekend,hour_sin,hour_cos,weekday_sin,weekday_cos,month_sin,month_cos,time_of_day_min,seconds_since_midnight,local_week,local_day_of_year,has_schedule,has_runway,has_stand,route,airport_hour,airport_weekday,aircraft_airport,runway_airport
u64,u64,i32,i32,i32,i32,i32,"datetime[μs, UTC]","datetime[μs, UTC]","datetime[μs, UTC]",i32,i32,i32,i32,i32,i64,i8,i8,i8,i8,i32,i64,i64,datetime[μs],i8,i8,i8,i16,i8,f64,f64,f64,f64,f64,f64,i8,i16,i8,i16,i8,i8,i8,i32,i32,i32,i32,i32
182908096,280366323,32374,1,6,1151,1,2025-01-01 00:35:08 UTC,2025-01-01 00:21:55 UTC,2025-01-01 00:20:00 UTC,62,48,414,793,1,793,0,0,0,0,1,115,-793,2025-01-01 01:21:55,1,3,1,1,0,0.258819,0.965926,0.433884,-0.900969,0.0,1.0,81,3635,1,1,1,1,1,3256,118,38,404,36
182971599,280366347,14295,1,5,1193,1,2025-01-01 00:38:39 UTC,2025-01-01 00:30:01 UTC,2025-01-01 00:30:00 UTC,15,6,263,518,1,518,0,0,0,0,1,1,-518,2025-01-01 01:30:01,1,3,1,1,0,0.258819,0.965926,0.433884,-0.900969,0.0,1.0,90,3609,1,1,1,1,1,2629,94,31,93,28
182772925,280351450,31067,1,10,1330,1,2025-01-01 00:14:00 UTC,2024-12-31 23:38:00 UTC,2024-12-31 23:45:00 UTC,16,44,814,2160,1,2160,0,0,0,0,1,-420,-2160,2025-01-01 02:38:00,2,3,1,1,0,0.5,0.866025,0.433884,-0.900969,0.0,1.0,-98,7176,1,1,1,1,1,5465,223,66,108,64
182773106,280351437,29159,1,10,1339,1,2025-01-01 00:05:56 UTC,2024-12-31 23:51:03 UTC,2024-12-31 23:45:00 UTC,24,44,796,893,1,893,0,0,0,0,1,363,-893,2025-01-01 02:51:03,2,3,1,1,0,0.5,0.866025,0.433884,-0.900969,0.0,1.0,-85,7191,1,1,1,1,1,5473,223,66,160,64
182773116,280351462,29210,1,10,1336,1,2025-01-01 00:24:56 UTC,2025-01-01 00:04:03 UTC,2024-12-31 23:50:00 UTC,24,44,934,1253,1,1253,0,0,0,0,1,843,-1253,2025-01-01 03:04:03,3,3,1,1,0,0.707107,0.707107,0.433884,-0.900969,0.0,1.0,-72,10787,1,1,1,1,1,5470,228,66,160,64
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
196716256,291241001,17054,1,6,1200,1,2025-09-30 22:01:45 UTC,2025-09-30 21:45:05 UTC,2025-09-30 21:45:00 UTC,65,49,473,1000,1,1000,0,0,0,0,1,5,-1000,2025-09-30 23:45:05,23,2,9,30,0,-0.258819,0.965926,0.974928,-0.222521,-0.866025,-0.5,-111,17153,40,273,1,1,1,3278,133,37,434,37
196132540,291240866,15099,1,8,1193,1,2025-09-30 22:01:03 UTC,2025-09-30 21:45:57 UTC,2025-09-30 21:30:00 UTC,24,19,565,906,1,906,0,0,0,0,1,957,-906,2025-09-30 23:45:57,23,2,9,30,0,-0.258819,0.965926,0.974928,-0.222521,-0.866025,-0.5,-111,17205,40,273,1,1,1,4121,181,51,158,48
196069988,291241019,11444,1,7,1151,1,2025-09-30 21:58:59 UTC,2025-09-30 21:47:01 UTC,2025-09-30 21:45:00 UTC,62,9,1277,718,1,718,0,0,0,0,1,121,-718,2025-09-30 23:47:01,23,2,9,30,0,-0.258819,0.965926,0.974928,-0.222521,-0.866025,-0.5,-109,17269,40,273,1,1,1,3738,157,44,405,38


## Modelling

In [ ]:
target = "TAXITIME_SEC_mvt"

drop_cols = [
    # Target / target reconstruction
    "TAXITIME_SEC_mvt",
    "reconstructed_taxi",
    "reconstructed_target",

    # QC generated from target
    "taxi_qc_class",
    "qc_class",
    "qc_negative_taxi",
    "qc_taxi_gt_1h",
    "qc_taxi_gt_2h",
    "qc_taxi_gt_24h",

    # Potentially target-derived timing variable
    "departure_delay_sec",

    # IDs
    "MVT_ID_mvt",
    "FLIGHT_ID_mvt",
    "FLIGHT_mvt",

    # Raw timestamps
    "MVT_TIME_UTC_mvt",
    "BLOCK_TIME_UTC_mvt",
    "SCHED_TIME_UTC_mvt",
    "local_datetime",
]

X_train = train_enc.drop(drop_cols)
y_train = train_enc[target]

X_valid = valid_enc.drop(drop_cols)
y_valid = valid_enc[target]

X_test = test_enc.drop(drop_cols)
y_test = test_enc[target]

print("TRAIN:", X_train.shape)
print("VALID:", X_valid.shape)
print("TEST :", X_test.shape)

TRAIN: (1553484, 30)
VALID: (344686, 30)
TEST : (164010, 30)


In [ ]:
print("Remaining non-numeric columns:")
print([
    (col, dtype)
    for col, dtype in zip(X_train.columns, X_train.dtypes)
    if dtype not in (
        pl.Int8, pl.Int16, pl.Int32, pl.Int64,
        pl.UInt8, pl.UInt16, pl.UInt32, pl.UInt64,
        pl.Float32, pl.Float64, pl.Boolean
    )
])

Remaining non-numeric columns:
[]


In [ ]:
#Models
ridge = Pipeline([
    ("scaler", RobustScaler()),
    ("model", Ridge(
        alpha=10.0,
        fit_intercept=True
    ))
])


lasso = Pipeline([
    ("scaler", RobustScaler()),
    ("model", Lasso(
        alpha=100.0,
        max_iter=1000,
        tol=1e-3,
        selection="random",
        random_state=42
    ))
])


elasticnet = Pipeline([
    ("scaler", RobustScaler()),
    ("model", ElasticNet(
        alpha=0.01,
        l1_ratio=0.5,
        fit_intercept=True,
        max_iter=1000,
        tol=1e-4
    ))
])


extra_trees = Pipeline([
    ("quantile", QuantileTransformer(
        n_quantiles=1000,
        output_distribution="normal",
        random_state=42
    )),
    ("model", ExtraTreesRegressor(
        n_estimators=500,
        max_depth=None,
        min_samples_split=5,
        min_samples_leaf=2,
        max_features=0.8,
        bootstrap=False,
        n_jobs=-1,
        random_state=42
    ))
])


xgb = Pipeline([
    ("quantile", QuantileTransformer(
        n_quantiles=1000,
        output_distribution="normal",
        random_state=42
    )),
    ("model", XGBRegressor(
        n_estimators=1000,
        learning_rate=0.05,
        max_depth=6,
        min_child_weight=5,
        subsample=0.8,
        colsample_bytree=0.8,
        gamma=0.0,
        reg_alpha=0.0,
        reg_lambda=1.0,
        objective="reg:squarederror",
        tree_method="hist",
        n_jobs=-1,
        random_state=42
    ))
])


lgbm = Pipeline([
    ("quantile", QuantileTransformer(
        n_quantiles=1000,
        output_distribution="normal",
        random_state=42
    )),
    ("model", LGBMRegressor(
        n_estimators=1000,
        learning_rate=0.05,
        num_leaves=31,
        max_depth=-1,
        min_child_samples=50,
        subsample=0.8,
        subsample_freq=1,
        colsample_bytree=0.8,
        reg_alpha=0.0,
        reg_lambda=1.0,
        objective="regression",
        n_jobs=-1,
        random_state=42,
        verbosity=-1
    ))
])


catboost = Pipeline([
    ("quantile", QuantileTransformer(
        n_quantiles=1000,
        output_distribution="normal",
        random_state=42
    )),
    ("model", CatBoostRegressor(
        iterations=1000,
        learning_rate=0.05,
        depth=8,
        l2_leaf_reg=5.0,
        loss_function="RMSE",
        random_seed=42,
        verbose=False,
        thread_count=-1
    ))
])

#Ensembled Model
stacking = VotingRegressor(
    estimators=[
    ("elasticnet", elasticnet),
    ("xgb", xgb),
    ("lgbm", lgbm),
    ("catboost", catboost)],
    n_jobs=-1,
    weights=[0.25, 0.3, 0.7, 0.5]
)

In [ ]:
class CPCV(BaseCrossValidator):

    def __init__(
        self,
        n_groups=10,
        n_test_groups=2,
        purge_minutes=60,
        embargo_minutes=30,
        time_col="local_datetime",
    ):
        self.n_groups = n_groups
        self.n_test_groups = n_test_groups
        self.purge_minutes = purge_minutes
        self.embargo_minutes = embargo_minutes
        self.time_col = time_col

    def get_n_splits(self, X=None, y=None, groups=None):
        from math import comb
        return comb(self.n_groups, self.n_test_groups)

    def split(self, X, y=None, groups=None):

        if not isinstance(X, pl.DataFrame):
            raise TypeError("CPCV expects a Polars DataFrame")

        if self.time_col not in X.columns:
            raise ValueError(
                f"{self.time_col!r} not found in DataFrame"
            )

        n = X.height

        if n < self.n_groups:
            raise ValueError(
                f"n_samples={n} is smaller than "
                f"n_groups={self.n_groups}"
            )

        # Original positional indices
        positions = np.arange(n)

        # Datetime values
        times = X.get_column(self.time_col).to_numpy()

        # Chronological ordering
        order = np.argsort(times)

        ordered_positions = positions[order]
        ordered_times = times[order]

        # Equal-sized chronological groups
        groups_idx = np.array_split(
            ordered_positions,
            self.n_groups,
        )

        # All combinations of test groups
        from itertools import combinations

        for test_groups in combinations(
            range(self.n_groups),
            self.n_test_groups,
        ):

            test_groups = tuple(test_groups)

            test_idx = np.concatenate(
                [groups_idx[g] for g in test_groups]
            )

            test_idx = np.sort(test_idx)

            test_times = times[test_idx]

            test_start = test_times.min()
            test_end = test_times.max()

            purge_delta = np.timedelta64(
                self.purge_minutes,
                "m",
            )

            embargo_delta = np.timedelta64(
                self.embargo_minutes,
                "m",
            )

            purge_start = test_start - purge_delta
            embargo_end = test_end + embargo_delta

            # Start with every observation not in test
            is_test = np.zeros(
                n,
                dtype=bool,
            )

            is_test[test_idx] = True

            # Purge observations immediately before test
            purge_mask = (
                (times >= purge_start)
                & (times < test_start)
            )

            # Embargo observations immediately after test
            embargo_mask = (
                (times > test_end)
                & (times <= embargo_end)
            )

            excluded = (
                is_test
                | purge_mask
                | embargo_mask
            )

            train_idx = positions[~excluded]

            train_idx = np.sort(train_idx)

            # Safety checks
            if len(train_idx) == 0:
                raise RuntimeError(
                    f"CPCV generated an empty training fold. "
                    f"Test groups={test_groups}, "
                    f"test_size={len(test_idx)}"
                )

            if len(test_idx) == 0:
                raise RuntimeError(
                    f"CPCV generated an empty test fold. "
                    f"Test groups={test_groups}"
                )

            yield train_idx, test_idx

In [ ]:
print("TARGET:", target)

print("\nX columns:")
print(X_train.columns)

print("\nPotential leakage:")
print([
    c for c in X_train.columns
    if any(k in c.lower() for k in [
        "taxi",
        "target",
        "reconstructed",
        "departure_delay"
    ])
])

TARGET: TAXITIME_SEC_mvt

X columns:
['FLIGHT_RULE_mvt', 'ADEP_mvt', 'ADES_mvt', 'PHASE_mvt', 'AIRCRAFT_TYPE_mvt', 'RUNWAY_mvt', 'STAND_mvt', 'local_hour', 'local_weekday', 'local_month', 'local_day', 'is_weekend', 'hour_sin', 'hour_cos', 'weekday_sin', 'weekday_cos', 'month_sin', 'month_cos', 'time_of_day_min', 'seconds_since_midnight', 'local_week', 'local_day_of_year', 'has_schedule', 'has_runway', 'has_stand', 'route', 'airport_hour', 'airport_weekday', 'aircraft_airport', 'runway_airport']

Potential leakage:
[]


In [ ]:
models = {
    "ridge": ridge,
    "lasso": lasso,
    "elasticnet": elasticnet,
    "xgb": xgb,
    "lgbm": lgbm,
    "catboost": catboost,
    "stacking": stacking,
}

mlflow.set_experiment("Model Experiments")

cv = CPCV(
    n_groups=5,
    n_test_groups=2,
    purge_minutes=60,
    embargo_minutes=30,
    time_col="local_datetime",
)

results = []
residuals = []

for name, model in models.items():

    with mlflow.start_run(run_name=f"{name}_Experiment"):

        fold_metrics = []
        fold_residuals = []

        for fold, (train_idx, test_idx) in enumerate(
            cv.split(train_enc)
        ):

            # Polars row indexing
            fold_train = train_enc[train_idx]
            fold_test = train_enc[test_idx]

            # Separate target
            y_train = fold_train[target]
            y_test = fold_test[target]

            X_train = fold_train.drop(drop_cols)
            X_test = fold_test.drop(drop_cols)

            # Safety check
            non_numeric = [
                (col, dtype)
                for col, dtype in zip(X_train.columns, X_train.dtypes)
                if dtype not in (
                    pl.Boolean,
                    pl.Int8, pl.Int16, pl.Int32, pl.Int64,
                    pl.UInt8, pl.UInt16, pl.UInt32, pl.UInt64,
                    pl.Float32, pl.Float64,
                )
            ]

            if non_numeric:
                raise TypeError(
                    f"Fold {fold} contains non-numeric columns: "
                    f"{non_numeric}"
                )

            # sklearn boundary
            X_train_np = X_train.to_numpy()
            X_test_np = X_test.to_numpy()

            y_train_np = y_train.to_numpy()
            y_test_np = y_test.to_numpy()

            estimator = clone(model)

            estimator.fit(
                X_train_np,
                y_train_np,
            )

            pred = estimator.predict(X_test_np)

            residuals = y_test_np - pred

            mae = mean_absolute_error(
                y_test_np,
                pred,
            )

            rmse = np.sqrt(
                mean_squared_error(
                    y_test_np,
                    pred,
                )
            )

            r2 = r2_score(
                y_test_np,
                pred,
            )

            fold_metrics.append({
                "fold": fold,
                "mae": mae,
                "rmse": rmse,
                "r2": r2,
            })

            fold_residuals.extend(
                residuals.tolist()
            )

            print(
                f"{name} | "
                f"Fold {fold:02d} | "
                f"MAE={mae:.2f} | "
                f"RMSE={rmse:.2f} | "
                f"R²={r2:.4f}"
            )

        # Aggregate CV metrics
        metrics_df = pl.DataFrame(fold_metrics)

        mean_mae = metrics_df["mae"].mean()
        std_mae = metrics_df["mae"].std()

        mean_rmse = metrics_df["rmse"].mean()
        std_rmse = metrics_df["rmse"].std()

        mean_r2 = metrics_df["r2"].mean()
        std_r2 = metrics_df["r2"].std()

        residuals_np = np.asarray(
            fold_residuals,
            dtype=np.float64,
        )

        print("\n" + "=" * 60)
        print(name)
        print("=" * 60)

        print(
            f"MAE  : {mean_mae:.2f} ± {std_mae:.2f}"
        )

        print(
            f"RMSE : {mean_rmse:.2f} ± {std_rmse:.2f}"
        )

        print(
            f"R²   : {mean_r2:.4f} ± {std_r2:.4f}"
        )

        print(
            f"Residual mean : {residuals_np.mean():.2f}"
        )

        print(
            f"Residual std  : {residuals_np.std():.2f}"
        )

        print(
            f"Residual median : "
            f"{np.median(residuals_np):.2f}"
        )

        print(
            f"Residual Q05 : "
            f"{np.quantile(residuals_np, 0.05):.2f}"
        )

        print(
            f"Residual Q95 : "
            f"{np.quantile(residuals_np, 0.95):.2f}"
        )

        print(
            f"Max |residual| : "
            f"{np.max(np.abs(residuals_np)):.2f}"
        )

        # MLflow
        mlflow.log_metrics({
            "cv_mae_mean": mean_mae,
            "cv_mae_std": std_mae,
            "cv_rmse_mean": mean_rmse,
            "cv_rmse_std": std_rmse,
            "cv_r2_mean": mean_r2,
            "cv_r2_std": std_r2,
            "residual_mean": residuals_np.mean(),
            "residual_std": residuals_np.std(),
            "residual_median": np.median(residuals_np),
            "residual_q05": np.quantile(residuals_np, 0.05),
            "residual_q95": np.quantile(residuals_np, 0.95),
            "max_abs_residual": np.max(
                np.abs(residuals_np)
            ),
        })

        print()

2026/09/17 13:41:27 INFO mlflow.store.db.utils: Creating initial MLflow database tables...
2026/09/17 13:41:27 INFO mlflow.store.db.utils: Updating database tables
2026/09/17 13:41:30 INFO mlflow.tracking.fluent: Experiment with name 'Model Experiments' does not exist. Creating a new experiment.


ridge | Fold 00 | MAE=282.12 | RMSE=399.83 | R²=0.0577
ridge | Fold 01 | MAE=284.10 | RMSE=403.88 | R²=0.0675
ridge | Fold 02 | MAE=292.55 | RMSE=428.18 | R²=0.0629
ridge | Fold 03 | MAE=288.65 | RMSE=411.67 | R²=0.0641
ridge | Fold 04 | MAE=274.11 | RMSE=364.72 | R²=0.0767
ridge | Fold 05 | MAE=280.90 | RMSE=390.18 | R²=0.0778
ridge | Fold 06 | MAE=277.18 | RMSE=371.81 | R²=0.0806
ridge | Fold 07 | MAE=282.62 | RMSE=395.11 | R²=0.0835
ridge | Fold 08 | MAE=277.69 | RMSE=377.26 | R²=0.0857
ridge | Fold 09 | MAE=287.43 | RMSE=415.33 | R²=0.0223

ridge
MAE  : 282.74 ± 5.67
RMSE : 395.80 ± 20.22
R²   : 0.0679 ± 0.0186
Residual mean : 12.51
Residual std  : 396.06
Residual median : -48.77
Residual Q05 : -472.68
Residual Q95 : 665.71
Max |residual| : 6205.27

lasso | Fold 00 | MAE=295.69 | RMSE=411.94 | R²=-0.0003
lasso | Fold 01 | MAE=298.48 | RMSE=418.25 | R²=-0.0000
lasso | Fold 02 | MAE=307.25 | RMSE=442.74 | R²=-0.0019
lasso | Fold 03 | MAE=301.80 | RMSE=425.61 | R²=-0.0003
lasso | Fold

/usr/local/lib/python3.13/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(


lgbm | Fold 00 | MAE=203.30 | RMSE=325.60 | R²=0.3751


/usr/local/lib/python3.13/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(


...

stacking | Fold 07 | MAE=194.40 | RMSE=303.65 | R²=0.4587


/usr/local/lib/python3.13/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(


stacking | Fold 08 | MAE=188.72 | RMSE=282.62 | R²=0.4869


/usr/local/lib/python3.13/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(


stacking | Fold 09 | MAE=198.03 | RMSE=313.16 | R²=0.4441

stacking
MAE  : 197.63 ± 7.46
RMSE : 308.58 ± 23.97
R²   : 0.4338 ± 0.0423
Residual mean : 10.24
Residual std  : 309.25
Residual median : -41.15
Residual Q05 : -330.31
Residual Q95 : 493.16
Max |residual| : 6214.09



In [ ]:
mlflow.set_tracking_uri("sqlite:////content/mlflow.db")

# Search for all runs in the experiment "Model Experiments"
runs_df = mlflow.search_runs(
    experiment_names=["Model Experiments"],
    output_format="pandas"
)

# Display all columns and rows without truncation
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', None)

display(runs_df)

,run_id,experiment_id,status,artifact_uri,start_time,end_time,metrics.residual_median,metrics.cv_mae_std,metrics.cv_rmse_std,metrics.cv_mae_mean,metrics.residual_mean,metrics.cv_rmse_mean,metrics.residual_q05,metrics.cv_r2_std,metrics.residual_q95,metrics.residual_std,metrics.max_abs_residual,metrics.cv_r2_mean,tags.mlflow.source.type,tags.mlflow.runName,tags.mlflow.source.name,tags.mlflow.user
0,7a060217d2d8477287a2c8929c8da0c6,1,FINISHED,/content/mlruns/1/7a060217d2d8477287a2c8929c8d...,2026-09-17 15:24:10.239000+00:00,2026-09-17 17:11:14.637000+00:00,-41.150257,7.460441,23.966335,197.629344,10.244909,308.581772,-330.310433,0.042310,493.157973,309.248608,6214.092541,0.433836,NOTEBOOK,stacking_Experiment,fileId=1dhb7Q85BbPot2i2xc4PQcCm8iNQyVsnJ,root
1,da67d99e3d854a099e7e50223c951e99,1,FINISHED,/content/mlruns/1/da67d99e3d854a099e7e50223c95...,2026-09-17 14:40:43.653000+00:00,2026-09-17 15:24:10.212000+00:00,-33.130487,7.529915,23.847158,196.004853,11.730229,307.466093,-343.429650,0.042839,484.068472,308.074051,6262.465501,0.437856,NOTEBOOK,catboost_Experiment,fileId=1dhb7Q85BbPot2i2xc4PQcCm8iNQyVsnJ,root
2,7f5b680417a94ed1ab3a996bcf95721a,1,FINISHED,/content/mlruns/1/7f5b680417a94ed1ab3a996bcf95...,2026-09-17 14:16:15.678000+00:00,2026-09-17 14:40:43.633000+00:00,-33.813120,7.926068,24.434906,195.143366,11.033301,306.345485,-342.953470,0.045282,481.588616,307.023100,6285.302942,0.441932,NOTEBOOK,lgbm_Experiment,fileId=1dhb7Q85BbPot2i2xc4PQcCm8iNQyVsnJ,root
3,1284a6f37281456d9d19201bc530a597,1,FINISHED,/content/mlruns/1/1284a6f37281456d9d19201bc530...,2026-09-17 13:53:31.034000+00:00,2026-09-17 14:16:15.654000+00:00,-32.195068,8.810985,24.516072,194.486296,10.469702,305.467296,-347.071213,0.045752,477.382599,306.172482,6197.161255,0.445108,NOTEBOOK,xgb_Experiment,fileId=1dhb7Q85BbPot2i2xc4PQcCm8iNQyVsnJ,root
4,a78de72cb0e9488faf89dbd18157921d,1,FINISHED,/content/mlruns/1/a78de72cb0e9488faf89dbd18157...,2026-09-17 13:42:28.755000+00:00,2026-09-17 13:53:31.017000+00:00,-62.361452,6.371892,19.439367,287.234436,4.828965,398.681138,-472.901962,0.009292,675.642655,399.078228,6205.230019,0.054157,NOTEBOOK,elasticnet_Experiment,fileId=1dhb7Q85BbPot2i2xc4PQcCm8iNQyVsnJ,root
5,3167bd79abab405783460bdfbe96200c,1,FINISHED,/content/mlruns/1/3167bd79abab405783460bdfbe96...,2026-09-17 13:41:56.389000+00:00,2026-09-17 13:42:28.739000+00:00,-72.981606,6.683241,18.649043,295.617003,-0.015177,410.060367,-504.833711,0.001040,695.740922,410.441852,6217.740922,-0.000844,NOTEBOOK,lasso_Experiment,fileId=1dhb7Q85BbPot2i2xc4PQcCm8iNQyVsnJ,root
6,ad0437892d904026b7c4e7ee53849296,1,FINISHED,/content/mlruns/1/ad0437892d904026b7c4e7ee5384...,2026-09-17 13:41:30.438000+00:00,2026-09-17 13:41:56.370000+00:00,-48.771210,5.672474,20.222445,282.735937,12.512514,395.795908,-472.677057,0.018622,665.708742,396.062988,6205.271306,0.067872,NOTEBOOK,ridge_Experiment,fileId=1dhb7Q85BbPot2i2xc4PQcCm8iNQyVsnJ,root


## Test Submission

In [ ]:
test = pl.read_csv("/content/drive/MyDrive/prc-2026-datasets/merged_prc_data_test.csv")
test_ids = test["MVT_ID_mvt"]

In [ ]:
def engineer_submission_features(df: pl.DataFrame) -> pl.DataFrame:

    df = df.clone()

    # ============================================================
    # DATETIME
    # ============================================================

    datetime_cols = [
        "MVT_TIME_UTC_mvt",
        "BLOCK_TIME_UTC_mvt",
        "SCHED_TIME_UTC_mvt",
        "LOBT_flt",
        "IOBT_flt",
        "EOBT_1_flt",
        "ARVT_1_flt",
        "AOBT_3_flt",
        "ARVT_3_flt",
    ]

    for col in datetime_cols:
        if col in df.columns:
            # Only convert if the column is currently a string type
            if df[col].dtype != pl.Datetime:
                df = df.with_columns(
                    pl.col(col)
                    .str.to_datetime(
                        strict=False,
                        format=None,
                        time_zone="UTC",
                    )
                    .alias(col)
                )

    # ============================================================
    # LOCAL DATETIME
    # ============================================================

    df = df.with_columns(
        pl.when(pl.col("ADEP_mvt") == "EDDF").then(
            pl.col("BLOCK_TIME_UTC_mvt").dt.convert_time_zone("Europe/Berlin").dt.replace_time_zone(None)
        )
        .when(pl.col("ADEP_mvt") == "EDDM").then(
            pl.col("BLOCK_TIME_UTC_mvt").dt.convert_time_zone("Europe/Berlin").dt.replace_time_zone(None)
        )
        .when(pl.col("ADEP_mvt") == "EGLL").then(
            pl.col("BLOCK_TIME_UTC_mvt").dt.convert_time_zone("Europe/London").dt.replace_time_zone(None)
        )
        .when(pl.col("ADEP_mvt") == "EHAM").then(
            pl.col("BLOCK_TIME_UTC_mvt").dt.convert_time_zone("Europe/Amsterdam").dt.replace_time_zone(None)
        )
        .when(pl.col("ADEP_mvt") == "LEBL").then(
            pl.col("BLOCK_TIME_UTC_mvt").dt.convert_time_zone("Europe/Madrid").dt.replace_time_zone(None)
        )
        .when(pl.col("ADEP_mvt") == "LEMD").then(
            pl.col("BLOCK_TIME_UTC_mvt").dt.convert_time_zone("Europe/Madrid").dt.replace_time_zone(None)
        )
        .when(pl.col("ADEP_mvt") == "LFPG").then(
            pl.col("BLOCK_TIME_UTC_mvt").dt.convert_time_zone("Europe/Paris").dt.replace_time_zone(None)
        )
        .when(pl.col("ADEP_mvt") == "LIRF").then(
            pl.col("BLOCK_TIME_UTC_mvt").dt.convert_time_zone("Europe/Rome").dt.replace_time_zone(None)
        )
        .when(pl.col("ADEP_mvt") == "LTFM").then(
            pl.col("BLOCK_TIME_UTC_mvt").dt.convert_time_zone("Europe/Istanbul").dt.replace_time_zone(None)
        )
        .when(pl.col("ADEP_mvt") == "LSZH").then(
            pl.col("BLOCK_TIME_UTC_mvt").dt.convert_time_zone("Europe/Zurich").dt.replace_time_zone(None)
        )
        .otherwise(pl.col("BLOCK_TIME_UTC_mvt").dt.convert_time_zone("UTC").dt.replace_time_zone(None))
        .alias("local_datetime")
    )


    # ============================================================
    # TEMPORAL FEATURES
    # ============================================================

    df = df.with_columns([
        pl.col("local_datetime").dt.hour().alias("local_hour"),
        pl.col("local_datetime").dt.weekday().alias("local_weekday"),
        pl.col("local_datetime").dt.month().alias("local_month"),
        pl.col("local_datetime").dt.day().alias("local_day"),
        pl.col("local_datetime").dt.ordinal_day().alias("local_day_of_year"),
        pl.col("local_datetime").dt.week().alias("local_week"),
    ])

    df = df.with_columns([
        (pl.col("local_weekday") >= 6)
        .cast(pl.Int8)
        .alias("is_weekend"),

        (
            pl.col("local_hour") * 60
            + pl.col("local_datetime").dt.minute()
        ).alias("time_of_day_min"),

        (
            pl.col("local_hour") * 3600
            + pl.col("local_datetime").dt.minute() * 60
            + pl.col("local_datetime").dt.second()
        ).alias("seconds_since_midnight"),
    ])

    # ============================================================
    # CYCLICAL FEATURES
    # ============================================================

    df = df.with_columns([
        (2 * 3.141592653589793 * pl.col("local_hour") / 24)
        .sin()
        .alias("hour_sin"),

        (2 * 3.141592653589793 * pl.col("local_hour") / 24)
        .cos()
        .alias("hour_cos"),

        (2 * 3.141592653589793 * pl.col("local_weekday") / 7)
        .sin()
        .alias("weekday_sin"),

        (2 * 3.141592653589793 * pl.col("local_weekday") / 7)
        .cos()
        .alias("weekday_cos"),

        (2 * 3.141592653589793 * pl.col("local_month") / 12)
        .sin()
        .alias("month_sin"),

        (2 * 3.141592653589793 * pl.col("local_month") / 12)
        .cos()
        .alias("month_cos"),
    ])

    # ============================================================
    # MISSINGNESS
    # ============================================================

    df = df.with_columns([
        pl.col("SCHED_TIME_UTC_mvt")
        .is_not_null()
        .cast(pl.Int8)
        .alias("has_schedule"),

        pl.col("RUNWAY_mvt")
        .is_not_null()
        .cast(pl.Int8)
        .alias("has_runway"),

        pl.col("STAND_mvt")
        .is_not_null()
        .cast(pl.Int8)
        .alias("has_stand"),
    ])

    # ============================================================
    # ROUTE
    # ============================================================

    df = df.with_columns(
        (
            pl.col("ADEP_mvt").fill_null("UNKNOWN")
            + "→"
            + pl.col("ADES_mvt").fill_null("UNKNOWN")
        ).alias("route")
    )

    # ============================================================
    # INTERACTIONS
    # ============================================================

    df = df.with_columns([
        (
            pl.col("ADEP_mvt").fill_null("UNKNOWN")
            + "_"
            + pl.col("local_hour").cast(pl.String)
        ).alias("airport_hour"),

        (
            pl.col("ADEP_mvt").fill_null("UNKNOWN")
            + "_"
            + pl.col("local_weekday").cast(pl.String)
        ).alias("airport_weekday"),

        (
            pl.col("AIRCRAFT_TYPE_mvt").fill_null("UNKNOWN")
            + "_"
            + pl.col("ADEP_mvt").fill_null("UNKNOWN")
        ).alias("aircraft_airport"),

        (
            pl.col("RUNWAY_mvt").fill_null("UNKNOWN")
            + "_"
            + pl.col("ADEP_mvt").fill_null("UNKNOWN")
        ).alias("runway_airport"),
    ])

    return df

In [ ]:
def encode_submission(
    df: pl.DataFrame,
    encoders: dict, # This is actually 'mappings' from encode_features
) -> pl.DataFrame:

    df = df.clone()
    exprs = []

    for col, mapping in encoders.items(): # Here 'mapping' is the dict (value: idx)
        if col not in df.columns:
            continue
        exprs.append(
            pl.col(col)
            .replace(mapping, default=0) # Apply the mapping directly
            .cast(pl.Int32)
            .alias(col)
        )

    # The binary columns are not part of the 'encoders' dict directly,
    # but need to be cast consistently as done in encode_features.
    binary_cols = [
        "qc_negative_taxi",
        "qc_taxi_gt_1h",
        "qc_taxi_gt_2h",
        "qc_taxi_gt_24h",
        "has_schedule",
        "has_runway",
        "has_stand",
        "is_weekend",
    ]

    for col in binary_cols:
        if col in df.columns:
            exprs.append(
                pl.col(col)
                .fill_null(False)
                .cast(pl.Int8)
                .alias(col)
            )

    return df.with_columns(exprs)

In [ ]:


test = engineer_submission_features(
    test
)

test = encode_submission(
    test,
    encoders
)

ComputeError: cannot compare string with numeric type (i32)

In [ ]:
drop_cols = [
    "local_datetime",
    "MVT_ID_mvt",
    "FLIGHT_ID_mvt",
    "FLIGHT_mvt",
    "MVT_TIME_UTC_mvt",
    "BLOCK_TIME_UTC_mvt",
    "SCHED_TIME_UTC_mvt",
    'ADEP_flt',
    'ADES_FILED_flt',
    'ADES_flt',
    'AIRCRAFT_OPERATOR_flt',
    'AIRCRAFT_TYPE_flt',
    'AOBT_3_flt',
    'ARVT_1_flt',
    'ARVT_3_flt',
    'CALLSIGN_flt',
    'EOBT_1_flt',
    'FLIGHT_RULE_flt',
    'FLIGHT_TYPE_flt',
    'IOBT_flt',
    'LOBT_flt',
    'MARKET_SEGMENT_flt',
]
test = test.drop(drop_cols)

ColumnNotFoundError: "local_datetime" not found

In [ ]:
test = test.drop('WK_TBL_CAT_flt')

In [ ]:
set(test.columns) - set(X_train.columns)

set()

In [ ]:
temporal_cols = [
    "local_hour",
    "local_weekday",
    "local_month",
    "local_day",
    "local_day_of_year",
    "local_week",
    "time_of_day_min",
    "seconds_since_midnight",
    "hour_sin",
    "hour_cos",
    "weekday_sin",
    "weekday_cos",
    "month_sin",
    "month_cos",
]

test = test.with_columns(
    [pl.col(c).fill_null(0).alias(c) for c in temporal_cols]
)

In [ ]:
print(
    test.select(
        [pl.col(c).null_count().alias(c) for c in temporal_cols]
    )
)

shape: (1, 14)
┌───────────┬───────────┬───────────┬───────────┬───┬───────────┬───────────┬───────────┬──────────┐
│ local_hou ┆ local_wee ┆ local_mon ┆ local_day ┆ … ┆ weekday_s ┆ weekday_c ┆ month_sin ┆ month_co │
│ r         ┆ kday      ┆ th        ┆ ---       ┆   ┆ in        ┆ os        ┆ ---       ┆ s        │
│ ---       ┆ ---       ┆ ---       ┆ u32       ┆   ┆ ---       ┆ ---       ┆ u32       ┆ ---      │
│ u32       ┆ u32       ┆ u32       ┆           ┆   ┆ u32       ┆ u32       ┆           ┆ u32      │
╞═══════════╪═══════════╪═══════════╪═══════════╪═══╪═══════════╪═══════════╪═══════════╪══════════╡
│ 0         ┆ 0         ┆ 0         ┆ 0         ┆ … ┆ 0         ┆ 0         ┆ 0         ┆ 0        │
└───────────┴───────────┴───────────┴───────────┴───┴───────────┴───────────┴───────────┴──────────┘


In [ ]:
xgb.fit(X_train.to_numpy(), y_train.to_numpy())
stacking.fit(X_train.to_numpy(), y_train.to_numpy())

VotingRegressor(estimators=[('elasticnet',
                             Pipeline(steps=[('scaler', RobustScaler()),
                                             ('model',
                                              ElasticNet(alpha=0.01))])),
                            ('xgb',
                             Pipeline(steps=[('quantile',
                                              QuantileTransformer(output_distribution='normal',
                                                                  random_state=42)),
                                             ('model',
                                              XGBRegressor(base_score=None,
                                                           booster=None,
                                                           callbacks=None,
                                                           colsample_bylevel=None,
                                                           colsample_bynode=None,
                                                           colsample_bytree=...
                                                            reg_lambda=1.0,
                                                            subsample=0.8,
                                                            subsample_freq=1,
                                                            verbosity=-1))])),
                            ('catboost',
                             Pipeline(steps=[('quantile',
                                              QuantileTransformer(output_distribution='normal',
                                                                  random_state=42)),
                                             ('model',
                                              CatBoostRegressor(depth=8, iterations=1000, l2_leaf_reg=5.0, learning_rate=0.05, loss_function='RMSE', random_seed=42, verbose=False))]))],
                n_jobs=-1, weights=[0.25, 0.3, 0.7, 0.5])

In [ ]:
xgb_preds = xgb.predict(test.to_numpy())
stacking_preds = stacking.predict(test.to_numpy())


submission_xgb = pd.DataFrame({
    "MVT_ID_mvt": test_ids,
    "TAXITIME_SEC_mvt": xgb_preds,
})
submission_stacking = pd.DataFrame({
    "MVT_ID_mvt": test_ids,
    "TAXITIME_SEC_mvt": stacking_preds,
})

/usr/local/lib/python3.13/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(


In [ ]:
submission_xgb.to_parquet("/content/drive/MyDrive/prc-2026-datasets/submission_xgb.parquet", index=False)
submission_stacking.to_parquet("/content/drive/MyDrive/prc-2026-datasets/submission_stacking.parquet", index=False)